# Volatility-Focused Supervised LSTM Autoencoder

This notebook adopts the Week 7 regime-aware supervised LSTM autoencoder and makes the risk layer explicit. Local research and volatile-regime comparisons are capped at `2022-12-31` to avoid tuning on 2023-2025 data. The objective is lower realized volatility and crash exposure, not maximum return.

In [1]:
# Bootstrap: works locally, in Colab, or when repo_root is set manually.
import os
import sys
import subprocess
import tempfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DEFAULT_REPO_URL = "https://github.com/halloyy/Portfolio-Optimization-Lib.git"


def _is_repo_root(path: Path) -> bool:
    path = Path(path).expanduser()
    return (path / "pyproject.toml").exists() and (path / "src" / "portfolio_toolkit").exists()


def _candidate_roots() -> list[Path]:
    candidates: list[Path] = []
    if "repo_root" in globals():
        candidates.append(Path(repo_root).expanduser())
    if os.environ.get("PORTFOLIO_REPO_ROOT"):
        candidates.append(Path(os.environ["PORTFOLIO_REPO_ROOT"]).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    candidates.extend(
        [
            Path("/content/Portfolio-Optimization-Lib"),
            Path("/content/Portfolio-Optimizer"),
            Path("/workspace/Portfolio-Optimization-Lib"),
            Path("/Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib"),
        ]
    )
    return candidates


def _find_repo_root() -> Path | None:
    for candidate in _candidate_roots():
        if _is_repo_root(candidate):
            return candidate.resolve()
    return None


repo_root = _find_repo_root()

if repo_root is None and IN_COLAB:
    repo_url = os.environ.get("PORTFOLIO_REPO_URL", DEFAULT_REPO_URL)
    clone_target = Path(os.environ.get("PORTFOLIO_REPO_ROOT", "/content/Portfolio-Optimization-Lib")).expanduser()
    if not clone_target.exists():
        subprocess.run(["git", "clone", repo_url, str(clone_target)], check=True)
    repo_root = clone_target.resolve() if _is_repo_root(clone_target) else None
    if repo_root is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root)], check=True)

if repo_root is None:
    raise RuntimeError(
        "Cannot find repo root. Run this notebook from inside Portfolio-Optimization-Lib, "
        "or set repo_root = Path('/path/to/Portfolio-Optimization-Lib') before this cell."
    )

os.chdir(repo_root)
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio_matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("repo_root =", repo_root)
print("python    =", sys.executable)

repo_root = /content/Portfolio-Optimization-Lib
python    = /usr/bin/python3


In [2]:
import json
import math
from dataclasses import replace
from datetime import date
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.covariance import LedoitWolf

from portfolio_toolkit import (
    PortfolioWeights,
    backtest_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_return_target,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    write_backtest_artifacts,
)
from portfolio_toolkit.config import dataset_identifier

warnings.filterwarnings("ignore")
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu128 | cuda: True


## Configuration

In [3]:
DATASET_NAME = os.environ.get("DATASET_NAME", "shared_set_2")
MODEL_NAME = "hannah_volatility_lstm_autoencoder"
RUN_NAME = "Hannah_Volatility_LSTM_AE"
HORIZON = 5
SEQ_LEN = 20
REBALANCE_FREQUENCY = "weekly"
RESEARCH_END = pd.Timestamp(os.environ.get("VOL_RESEARCH_END", "2022-12-31"))

LATENT_DIM = 32
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = int(os.environ.get("LSTM_AE_EPOCHS", "18"))
DATES_PER_BATCH = 16
PATIENCE = 5
MASK_RATE = 0.10
GRAD_CLIP = 1.0

# Volatility-first loss profile. Alpha remains present, but risk heads carry more weight than Week 7.
RECON_WEIGHT = 1.00
RETURN_WEIGHT = 0.15
ALPHA_WEIGHT = 1.20
RANK_WEIGHT = 0.20
VOL_WEIGHT = 0.85
DOWNSIDE_WEIGHT = 0.95
TAIL_WEIGHT = 0.70
REGIME_WEIGHT = 0.40
HIGH_RISK_SAMPLE_MULTIPLIER = 1.75

NORMAL_MAX_WEIGHT = 0.10
HIGH_VOL_MAX_WEIGHT_OPTIONS = [0.03, 0.05, 0.075, 0.10]
BASE_TOP_FRACTION = 0.65
HIGH_RISK_MIN_ACTIVE_NAMES = 35
NORMAL_MIN_ACTIVE_NAMES = 18
TURNOVER_BLEND = 0.45
BETA_TARGET = 0.85
COV_LOOKBACK_DAYS = 252
MINVAR_TILT_NORMAL = 0.30
MINVAR_TILT_HIGH_RISK = 0.10

DOWNSIDE_SCORE_PENALTY = 0.75
TAIL_SCORE_PENALTY = 1.25
UNCERTAINTY_SCORE_PENALTY = 0.35
REGIME_SCORE_BONUS = 0.08

REGIME_CLASSES = ["calm_risk_on", "high_vol", "drawdown", "rebound"]
REGIME_CLASS_TO_ID = {name: idx for idx, name in enumerate(REGIME_CLASSES)}

FAST_SMOKE = os.environ.get("FAST_SMOKE", "0") == "1"
RANDOM_SEED = 99
LOG_TO_MLFLOW = os.environ.get("SKIP_MLFLOW", "0") != "1"

output_dir = repo_root / "runs" / "supervised_lstm_autoencoder_volatility"
output_dir.mkdir(parents=True, exist_ok=True)
NOTEBOOK_PATH = repo_root / "MODELS" / "Hannah" / "supervised_lstm_autoencoder_volatility.ipynb"

spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)
research_spec = replace(
    spec,
    name=f"{spec.name}_through_2022",
    end_date=RESEARCH_END.date(),
    test_end=RESEARCH_END.date(),
    dataset_id=f"{DATASET_NAME}_through_2022",
)
tradable_tickers = [ticker for ticker in research_spec.tickers if ticker != research_spec.benchmark_ticker]
backtest_spec = replace(research_spec, tickers=tradable_tickers)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True

print("Dataset preset:", DATASET_NAME)
print("Research end cap:", RESEARCH_END.date())
print("Research dataset id:", dataset_identifier(research_spec, repo_root=repo_root))
print("Tradable tickers:", len(tradable_tickers))
print("Benchmark ticker:", research_spec.benchmark_ticker)
print("Train/Val/Test windows:", research_spec.train_start, research_spec.train_end, research_spec.val_start, research_spec.val_end, research_spec.test_start, research_spec.test_end)
print("Epochs:", EPOCHS, "| FAST_SMOKE:", 1, "| MLflow logging:", 0)

Dataset preset: shared_set_2
Research end cap: 2022-12-31
Research dataset id: shared_set_2_through_2022
Tradable tickers: 25
Benchmark ticker: SPY
Train/Val/Test windows: 2014-01-02 2019-12-31 2020-01-02 2021-12-31 2022-01-03 2022-12-31
Epochs: 18 | FAST_SMOKE: 1 | MLflow logging: 0


## Volatility And Stress Features

In [4]:
BASE_FEATURE_NAMES = [
    "return_1d", "return_5d", "return_20d", "return_60d",
    "vol_5d", "vol_20d", "vol_60d", "downside_vol_20d", "atr_14",
    "momentum_5d", "momentum_20d", "momentum_60d", "momentum_120d",
    "rsi_14", "macd_hist", "bollinger_z_20d",
    "price_to_sma_20d", "price_to_sma_50d", "price_to_sma_200d",
    "volume_zscore_20d", "dollar_volume_ratio_20d", "intraday_range",
    "beta_20d_spy", "beta_60d_spy",
    "excess_return_20d_vs_spy", "excess_return_60d_vs_spy", "relative_momentum_20d_vs_spy",
]

CUSTOM_FEATURE_NAMES = [
    "mom_vol_ratio_60d", "trend_spread_20_50", "trend_spread_50_200", "rolling_sharpe_60d", "drawdown_60d",
    "spy_vol_20d_ann", "spy_momentum_20d", "spy_momentum_60d", "spy_drawdown_60d", "market_breadth_20d",
    "pct_above_sma50", "pct_above_sma200", "pct_20d_highs", "pct_20d_lows", "avg_universe_drawdown_60d",
    "cross_sectional_dispersion_20d", "gap_frequency_20d", "volume_shock_20d", "abnormal_range_20d",
    "vol_compression_ratio_20_60", "distance_from_60d_high", "momentum_x_spy_vol", "momentum_x_spy_drawdown",
    "beta_instability_60d",
    "avg_pairwise_corr_20d", "avg_pairwise_corr_60d", "corr_breakdown_20_60",
    "breadth_deterioration_20d", "dollar_volume_shock_20d", "abnormal_range_persistence_10d",
    "liquidity_stress_20d", "benchmark_agnostic_stress", "beta_x_corr_stress",
]

ALL_FEATURE_NAMES = BASE_FEATURE_NAMES + CUSTOM_FEATURE_NAMES
REGIME_CONTEXT_COLUMNS = ["spy_vol_20d_ann", "spy_momentum_20d", "spy_momentum_60d", "spy_drawdown_60d"]


def _safe_pct_change(series: pd.Series, periods: int = 1) -> pd.Series:
    return series.astype(float).pct_change(periods)


def _rolling_average_pairwise_corr(return_wide: pd.DataFrame, window: int, min_periods: int) -> pd.Series:
    rows = []
    index = return_wide.index
    values = return_wide.to_numpy(dtype=float)
    for end in range(len(index)):
        start = max(0, end - window + 1)
        block = values[start : end + 1]
        valid_cols = np.isfinite(block).sum(axis=0) >= min_periods
        if valid_cols.sum() < 2:
            rows.append(np.nan)
            continue
        corr = np.corrcoef(block[:, valid_cols], rowvar=False)
        if corr.ndim != 2:
            rows.append(np.nan)
            continue
        mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
        rows.append(float(np.nanmean(corr[mask])))
    return pd.Series(rows, index=index)


def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    """Rebuild the exact volatility-focused LSTM-AE feature table from raw long-form prices."""
    frame = build_features(prices, feature_names=BASE_FEATURE_NAMES)
    panel = prices.copy()
    panel["date"] = pd.to_datetime(panel["date"], utc=True).dt.tz_localize(None)
    panel["ticker"] = panel["ticker"].astype(str).str.upper()
    panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
    eps = 1e-6

    frame["mom_vol_ratio_60d"] = frame["momentum_60d"] / (frame["vol_60d"].abs() + eps)
    frame["trend_spread_20_50"] = frame["price_to_sma_20d"] - frame["price_to_sma_50d"]
    frame["trend_spread_50_200"] = frame["price_to_sma_50d"] - frame["price_to_sma_200d"]
    frame["rolling_sharpe_60d"] = frame["return_60d"] / (frame["vol_60d"].abs() * np.sqrt(60.0) + eps)

    grouped_close = panel.groupby("ticker", sort=False)["adj_close"]
    rolling_high_20 = grouped_close.transform(lambda s: s.rolling(20, min_periods=20).max())
    rolling_low_20 = grouped_close.transform(lambda s: s.rolling(20, min_periods=20).min())
    rolling_high_60 = grouped_close.transform(lambda s: s.rolling(60, min_periods=20).max())
    prev_close = grouped_close.shift(1)

    event = panel.loc[:, ["date", "ticker"]].copy()
    event["drawdown_60d"] = panel["adj_close"] / rolling_high_60.replace(0.0, np.nan) - 1.0
    event["distance_from_60d_high"] = panel["adj_close"] / rolling_high_60.replace(0.0, np.nan) - 1.0
    event["gap_abs"] = (panel["open"] / prev_close.replace(0.0, np.nan) - 1.0).abs()
    event["range_ratio"] = (panel["high"] - panel["low"]) / panel["adj_close"].replace(0.0, np.nan)
    event["gap_frequency_20d"] = event.groupby(panel["ticker"], sort=False)["gap_abs"].transform(lambda s: (s > 0.03).rolling(20, min_periods=10).mean())
    event["abnormal_range_20d"] = event["range_ratio"] / (event.groupby(panel["ticker"], sort=False)["range_ratio"].transform(lambda s: s.rolling(20, min_periods=10).mean()) + eps)
    event["abnormal_range_persistence_10d"] = event.groupby(panel["ticker"], sort=False)["abnormal_range_20d"].transform(lambda s: s.rolling(10, min_periods=5).mean())
    rolling_vol_20 = grouped_close.transform(lambda s: _safe_pct_change(s).rolling(20, min_periods=20).std(ddof=0))
    rolling_vol_60 = grouped_close.transform(lambda s: _safe_pct_change(s).rolling(60, min_periods=30).std(ddof=0))
    event["vol_compression_ratio_20_60"] = rolling_vol_20 / (rolling_vol_60 + eps)
    dollar_volume = panel["adj_close"] * panel["volume"]
    dollar_volume_ma_20 = dollar_volume.groupby(panel["ticker"], sort=False).transform(lambda s: s.rolling(20, min_periods=10).mean())
    volume_ma_20 = panel.groupby("ticker", sort=False)["volume"].transform(lambda s: s.rolling(20, min_periods=10).mean())
    event["volume_shock_20d"] = panel["volume"] / volume_ma_20.replace(0.0, np.nan)
    event["dollar_volume_shock_20d"] = dollar_volume / dollar_volume_ma_20.replace(0.0, np.nan)
    event["liquidity_stress_20d"] = event["abnormal_range_persistence_10d"] / (event["dollar_volume_shock_20d"].clip(lower=0.10) + eps)
    event = event.drop(columns=["gap_abs", "range_ratio"])
    frame = frame.merge(event, on=["date", "ticker"], how="left")

    spy = panel.loc[panel["ticker"] == "SPY", ["date", "adj_close"]].sort_values("date").copy()
    if spy.empty:
        regime = pd.DataFrame({"date": pd.to_datetime(panel["date"].drop_duplicates())})
        for col in REGIME_CONTEXT_COLUMNS:
            regime[col] = np.nan
    else:
        spy_ret = spy["adj_close"].pct_change()
        regime = spy.loc[:, ["date"]].copy()
        regime["spy_vol_20d_ann"] = spy_ret.rolling(20, min_periods=20).std(ddof=0) * np.sqrt(252.0)
        regime["spy_momentum_20d"] = spy["adj_close"].pct_change(20)
        regime["spy_momentum_60d"] = spy["adj_close"].pct_change(60)
        spy_high_60 = spy["adj_close"].rolling(60, min_periods=20).max()
        regime["spy_drawdown_60d"] = spy["adj_close"] / spy_high_60.replace(0.0, np.nan) - 1.0
    frame = frame.merge(regime, on="date", how="left")

    non_benchmark = panel.loc[panel["ticker"] != "SPY"].copy()
    nb_close = non_benchmark.groupby("ticker", sort=False)["adj_close"]
    nb_return = nb_close.pct_change()
    sma_20 = nb_close.transform(lambda s: s.rolling(20, min_periods=20).mean())
    sma_50 = nb_close.transform(lambda s: s.rolling(50, min_periods=30).mean())
    sma_200 = nb_close.transform(lambda s: s.rolling(200, min_periods=100).mean())
    high_20 = nb_close.transform(lambda s: s.rolling(20, min_periods=20).max())
    low_20 = nb_close.transform(lambda s: s.rolling(20, min_periods=20).min())
    high_60_nb = nb_close.transform(lambda s: s.rolling(60, min_periods=20).max())
    breadth_panel = non_benchmark.loc[:, ["date", "ticker"]].copy()
    breadth_panel["above_sma20"] = non_benchmark["adj_close"] > sma_20
    breadth_panel["above_sma50"] = non_benchmark["adj_close"] > sma_50
    breadth_panel["above_sma200"] = non_benchmark["adj_close"] > sma_200
    breadth_panel["is_20d_high"] = non_benchmark["adj_close"] >= high_20
    breadth_panel["is_20d_low"] = non_benchmark["adj_close"] <= low_20
    breadth_panel["universe_drawdown_60d"] = non_benchmark["adj_close"] / high_60_nb.replace(0.0, np.nan) - 1.0
    breadth_panel["daily_return"] = nb_return.to_numpy(dtype=float)

    breadth = breadth_panel.groupby("date").agg(
        market_breadth_20d=("above_sma20", "mean"),
        pct_above_sma50=("above_sma50", "mean"),
        pct_above_sma200=("above_sma200", "mean"),
        pct_20d_highs=("is_20d_high", "mean"),
        pct_20d_lows=("is_20d_low", "mean"),
        avg_universe_drawdown_60d=("universe_drawdown_60d", "mean"),
        cross_sectional_dispersion_20d=("daily_return", "std"),
    ).reset_index()
    breadth["cross_sectional_dispersion_20d"] = breadth["cross_sectional_dispersion_20d"].rolling(20, min_periods=10).mean()
    breadth["breadth_deterioration_20d"] = breadth["market_breadth_20d"] - breadth["market_breadth_20d"].shift(20)

    ret_wide = non_benchmark.pivot(index="date", columns="ticker", values="adj_close").sort_index().pct_change()
    corr = pd.DataFrame({
        "date": ret_wide.index,
        "avg_pairwise_corr_20d": _rolling_average_pairwise_corr(ret_wide, 20, 12).to_numpy(dtype=float),
        "avg_pairwise_corr_60d": _rolling_average_pairwise_corr(ret_wide, 60, 30).to_numpy(dtype=float),
    })
    corr["corr_breakdown_20_60"] = corr["avg_pairwise_corr_20d"] - corr["avg_pairwise_corr_60d"]
    breadth = breadth.merge(corr, on="date", how="left")
    breadth["benchmark_agnostic_stress"] = (
        breadth["cross_sectional_dispersion_20d"].rank(pct=True)
        + (-breadth["avg_universe_drawdown_60d"]).rank(pct=True)
        + breadth["avg_pairwise_corr_20d"].rank(pct=True)
        + (-breadth["breadth_deterioration_20d"]).rank(pct=True)
    ) / 4.0
    frame = frame.merge(breadth, on="date", how="left")

    frame["momentum_x_spy_vol"] = frame["momentum_20d"] * frame["spy_vol_20d_ann"]
    frame["momentum_x_spy_drawdown"] = frame["momentum_20d"] * frame["spy_drawdown_60d"]
    frame["beta_instability_60d"] = frame.groupby("ticker", sort=False)["beta_20d_spy"].transform(lambda s: s.rolling(60, min_periods=20).std(ddof=0))
    frame["beta_x_corr_stress"] = frame["beta_60d_spy"].abs() * frame["avg_pairwise_corr_20d"]

    return frame.loc[:, ["date", "ticker"] + ALL_FEATURE_NAMES]

In [5]:
prices_all = load_prices(DATASET_NAME, repo_root=repo_root)
prices_all["date"] = pd.to_datetime(prices_all["date"], utc=True).dt.tz_localize(None)
prices = prices_all.loc[prices_all["date"] <= RESEARCH_END].copy()

# Make the standard toolkit backtester use the pre-2023 research data instead of the full cached dataset.
research_cache_path = repo_root / "data_cache" / f"{dataset_identifier(research_spec, repo_root=repo_root)}.parquet"
research_cache_path.parent.mkdir(parents=True, exist_ok=True)
prices.to_parquet(research_cache_path, index=False)
print("Wrote research cache:", research_cache_path)
print("Price frame shape:", prices.shape)
print("Date range:", prices["date"].min(), "->", prices["date"].max())
print("Unique tickers:", prices["ticker"].nunique())

feature_frame = build_model_features(prices)
print("Feature frame shape:", feature_frame.shape)
print("Feature count:", len(ALL_FEATURE_NAMES))
display(feature_frame.head())

Wrote research cache: /content/Portfolio-Optimization-Lib/data_cache/shared_set_2_through_2022.parquet
Price frame shape: (58916, 8)
Date range: 2014-01-02 00:00:00 -> 2022-12-30 00:00:00
Unique tickers: 26
Feature frame shape: (58916, 62)
Feature count: 60


,date,ticker,return_1d,return_5d,return_20d,return_60d,vol_5d,vol_20d,vol_60d,downside_vol_20d,...,beta_instability_60d,avg_pairwise_corr_20d,avg_pairwise_corr_60d,corr_breakdown_20_60,breadth_deterioration_20d,dollar_volume_shock_20d,abnormal_range_persistence_10d,liquidity_stress_20d,benchmark_agnostic_stress,beta_x_corr_stress
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-01-03,AAPL,-0.021966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-01-06,AAPL,0.005453,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2014-01-07,AAPL,-0.007152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-01-08,AAPL,0.006333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Tail Targets, Regimes, And Sequences

In [6]:
return_target_col = f"forward_return_{HORIZON}d"
alpha_target_col = f"forward_alpha_{HORIZON}d_vs_spy"
forward_vol_target_col = f"forward_realized_vol_{HORIZON}d"
bottom_quintile_col = f"bottom_return_quintile_{HORIZON}d"
tail_loss_target_col = f"forward_tail_loss_{HORIZON}d"


def make_forward_risk_targets(prices: pd.DataFrame, horizon: int) -> pd.DataFrame:
    panel = prices.copy()
    panel["date"] = pd.to_datetime(panel["date"], utc=True).dt.tz_localize(None)
    panel["ticker"] = panel["ticker"].astype(str).str.upper()
    panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
    panel["daily_return"] = panel.groupby("ticker", sort=False)["adj_close"].pct_change()

    def _future_vol(ret: pd.Series) -> pd.Series:
        shifted = ret.shift(-1)
        return shifted.iloc[::-1].rolling(horizon, min_periods=horizon).std(ddof=0).iloc[::-1] * np.sqrt(252.0)

    def _future_tail_loss(ret: pd.Series) -> pd.Series:
        shifted = ret.shift(-1)
        worst = shifted.iloc[::-1].rolling(horizon, min_periods=horizon).min().iloc[::-1]
        return (-worst).clip(lower=0.0)

    panel[forward_vol_target_col] = panel.groupby("ticker", sort=False)["daily_return"].transform(_future_vol)
    panel[tail_loss_target_col] = panel.groupby("ticker", sort=False)["daily_return"].transform(_future_tail_loss)
    return panel.loc[:, ["date", "ticker", forward_vol_target_col, tail_loss_target_col]]


def make_regime_thresholds(feature_frame: pd.DataFrame, train_start, train_end) -> dict[str, float]:
    context_cols = ["date", *REGIME_CONTEXT_COLUMNS, "benchmark_agnostic_stress", "breadth_deterioration_20d"]
    context = feature_frame.loc[:, context_cols].drop_duplicates("date").copy()
    dates = pd.to_datetime(context["date"])
    train_context = context.loc[(dates >= pd.Timestamp(train_start)) & (dates <= pd.Timestamp(train_end))].replace([np.inf, -np.inf], np.nan)
    return {
        "high_vol_spy_vol_20d_ann": float(train_context["spy_vol_20d_ann"].quantile(0.70)),
        "drawdown_spy_drawdown_60d": float(min(-0.04, train_context["spy_drawdown_60d"].quantile(0.25))),
        "rebound_spy_momentum_20d": float(max(0.0, train_context["spy_momentum_20d"].median())),
        "stress_benchmark_agnostic": float(train_context["benchmark_agnostic_stress"].quantile(0.70)),
        "breadth_deterioration_20d": float(train_context["breadth_deterioration_20d"].quantile(0.25)),
    }


def assign_market_regime(frame: pd.DataFrame, thresholds: dict[str, float]) -> pd.Series:
    vol = pd.to_numeric(frame["spy_vol_20d_ann"], errors="coerce")
    drawdown = pd.to_numeric(frame["spy_drawdown_60d"], errors="coerce")
    momentum = pd.to_numeric(frame["spy_momentum_20d"], errors="coerce")
    agnostic_stress = pd.to_numeric(frame["benchmark_agnostic_stress"], errors="coerce")
    breadth_drop = pd.to_numeric(frame["breadth_deterioration_20d"], errors="coerce")
    labels = pd.Series("calm_risk_on", index=frame.index, dtype="object")
    labels.loc[(vol >= thresholds["high_vol_spy_vol_20d_ann"]) | (agnostic_stress >= thresholds["stress_benchmark_agnostic"])] = "high_vol"
    labels.loc[(drawdown <= thresholds["drawdown_spy_drawdown_60d"]) | (breadth_drop <= thresholds["breadth_deterioration_20d"])] = "drawdown"
    rebound = (drawdown <= thresholds["drawdown_spy_drawdown_60d"]) & (momentum > thresholds["rebound_spy_momentum_20d"])
    labels.loc[rebound] = "rebound"
    return labels


return_targets = make_forward_return_target(prices, horizon=HORIZON)
alpha_targets = make_forward_alpha_target(prices, horizon=HORIZON)
risk_targets = make_forward_risk_targets(prices, horizon=HORIZON)
regime_thresholds = make_regime_thresholds(feature_frame, research_spec.train_start, research_spec.train_end)

feature_frame = feature_frame.copy()
feature_frame["market_regime"] = assign_market_regime(feature_frame, regime_thresholds)
feature_frame["market_regime_code"] = feature_frame["market_regime"].map(REGIME_CLASS_TO_ID).astype(int)
feature_frame["is_high_risk_regime"] = feature_frame["market_regime"].isin(["high_vol", "drawdown"]).astype(float)

target_frame = feature_frame.merge(return_targets, on=["date", "ticker"], how="left")
target_frame = target_frame.merge(alpha_targets, on=["date", "ticker"], how="left")
target_frame = target_frame.merge(risk_targets, on=["date", "ticker"], how="left")
target_frame = target_frame.replace([np.inf, -np.inf], np.nan)
target_frame[bottom_quintile_col] = (target_frame.groupby("date")[return_target_col].rank(method="average", pct=True) <= 0.20).astype(float)
target_frame = (
    target_frame.dropna(subset=ALL_FEATURE_NAMES + [return_target_col, alpha_target_col, forward_vol_target_col, tail_loss_target_col])
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)
target_frame["historical_vol_20d"] = np.clip(target_frame["vol_20d"].to_numpy(dtype=float) * np.sqrt(252.0), 1e-4, None)
target_frame[forward_vol_target_col] = np.clip(target_frame[forward_vol_target_col].to_numpy(dtype=float), 1e-4, None)
target_frame[tail_loss_target_col] = np.clip(target_frame[tail_loss_target_col].to_numpy(dtype=float), 0.0, None)
target_frame["forward_vol_log"] = np.log1p(target_frame[forward_vol_target_col])
target_frame["tail_loss_log"] = np.log1p(target_frame[tail_loss_target_col])


def _split_mask(frame: pd.DataFrame, start, end) -> pd.Series:
    dates = pd.to_datetime(frame["date"])
    return (dates >= pd.Timestamp(start)) & (dates <= pd.Timestamp(end))


train_rows = target_frame.loc[_split_mask(target_frame, research_spec.train_start, research_spec.train_end)].copy()
val_rows = target_frame.loc[_split_mask(target_frame, research_spec.val_start, research_spec.val_end)].copy()
test_rows = target_frame.loc[_split_mask(target_frame, research_spec.test_start, research_spec.test_end)].copy()
train_model_rows = train_rows.loc[train_rows["ticker"] != research_spec.benchmark_ticker].copy()

train_means = train_model_rows[ALL_FEATURE_NAMES].mean()
train_stds = train_model_rows[ALL_FEATURE_NAMES].std(ddof=0).replace(0.0, 1.0)

target_means = {
    "return": float(train_model_rows[return_target_col].mean()),
    "alpha": float(train_model_rows[alpha_target_col].mean()),
    "vol_log": float(train_model_rows["forward_vol_log"].mean()),
    "tail_log": float(train_model_rows["tail_loss_log"].mean()),
}
target_stds = {
    "return": float(train_model_rows[return_target_col].std(ddof=0) or 1.0),
    "alpha": float(train_model_rows[alpha_target_col].std(ddof=0) or 1.0),
    "vol_log": float(train_model_rows["forward_vol_log"].std(ddof=0) or 1.0),
    "tail_log": float(train_model_rows["tail_loss_log"].std(ddof=0) or 1.0),
}


def standardize_features(frame: pd.DataFrame) -> np.ndarray:
    return ((frame[ALL_FEATURE_NAMES] - train_means) / train_stds).to_numpy(dtype=np.float32)


X_all = standardize_features(target_frame)

print("Modeling frame:", target_frame.shape)
print("Train/Val/Test rows:", len(train_rows), len(val_rows), len(test_rows))
print("Train rows excluding benchmark:", len(train_model_rows))
print("Feature count:", len(ALL_FEATURE_NAMES))
print("Regime thresholds:", json.dumps(regime_thresholds, indent=2, sort_keys=True))
print("Regime distribution by unique date:")
display(target_frame.drop_duplicates(["date"])["market_regime"].value_counts(normalize=True).rename("share").to_frame())

Modeling frame: (53607, 73)
Train/Val/Test rows: 34081 13130 6396
Train rows excluding benchmark: 32770
Feature count: 60
Regime thresholds: {
  "breadth_deterioration_20d": -0.28,
  "drawdown_spy_drawdown_60d": -0.04,
  "high_vol_spy_vol_20d_ann": 0.13226231138990618,
  "rebound_spy_momentum_20d": 0.012521434140163934,
  "stress_benchmark_agnostic": 0.5386325472997403
}
Regime distribution by unique date:


,share
market_regime,
calm_risk_on,0.407371
drawdown,0.359360
high_vol,0.171678
rebound,0.061591


In [7]:
def make_model_sequences(frame: pd.DataFrame, X: np.ndarray, seq_len: int) -> tuple[np.ndarray, pd.DataFrame]:
    Xs: list[np.ndarray] = []
    meta: list[dict[str, object]] = []
    for ticker, group in frame.groupby("ticker", sort=False):
        positions = group.index.to_numpy(dtype=int)
        if len(positions) < seq_len:
            continue
        dates = pd.to_datetime(group["date"]).to_numpy()
        for end_offset in range(seq_len - 1, len(positions)):
            window_positions = positions[end_offset - seq_len + 1 : end_offset + 1]
            row = group.iloc[end_offset]
            Xs.append(X[window_positions])
            meta.append(
                {
                    "date": pd.Timestamp(dates[end_offset]),
                    "ticker": ticker,
                    "row_position": int(positions[end_offset]),
                    "target_return": float(row[return_target_col]),
                    "target_alpha": float(row[alpha_target_col]),
                    "target_forward_vol": float(row[forward_vol_target_col]),
                    "target_tail_loss": float(row[tail_loss_target_col]),
                    "target_vol_log": float(row["forward_vol_log"]),
                    "target_tail_log": float(row["tail_loss_log"]),
                    "target_bottom_quintile": float(row[bottom_quintile_col]),
                    "market_regime": str(row["market_regime"]),
                    "market_regime_code": int(row["market_regime_code"]),
                    "is_high_risk_regime": float(row["is_high_risk_regime"]),
                    "historical_vol_20d": float(row["historical_vol_20d"]),
                    "spy_vol_20d_ann": float(row["spy_vol_20d_ann"]),
                    "spy_drawdown_60d": float(row["spy_drawdown_60d"]),
                    "spy_momentum_20d": float(row["spy_momentum_20d"]),
                    "beta_60d_spy": float(row["beta_60d_spy"]),
                    "benchmark_agnostic_stress": float(row["benchmark_agnostic_stress"]),
                    "avg_pairwise_corr_20d": float(row["avg_pairwise_corr_20d"]),
                }
            )
    return np.stack(Xs).astype(np.float32), pd.DataFrame(meta)


X_seq, seq_meta = make_model_sequences(target_frame, X_all, SEQ_LEN)
seq_meta["return_scaled"] = ((seq_meta["target_return"] - target_means["return"]) / target_stds["return"]).astype(np.float32)
seq_meta["alpha_scaled"] = ((seq_meta["target_alpha"] - target_means["alpha"]) / target_stds["alpha"]).astype(np.float32)
seq_meta["vol_scaled"] = ((seq_meta["target_vol_log"] - target_means["vol_log"]) / target_stds["vol_log"]).astype(np.float32)
seq_meta["tail_scaled"] = ((seq_meta["target_tail_log"] - target_means["tail_log"]) / target_stds["tail_log"]).astype(np.float32)
seq_meta["sample_weight"] = 1.0 + (HIGH_RISK_SAMPLE_MULTIPLIER - 1.0) * seq_meta["is_high_risk_regime"].astype(float)

non_benchmark_seq = seq_meta["ticker"] != research_spec.benchmark_ticker
train_seq_mask = _split_mask(seq_meta, research_spec.train_start, research_spec.train_end) & non_benchmark_seq
val_seq_mask = _split_mask(seq_meta, research_spec.val_start, research_spec.val_end) & non_benchmark_seq
test_seq_mask = _split_mask(seq_meta, research_spec.test_start, research_spec.test_end) & non_benchmark_seq

X_train_seq = X_seq[train_seq_mask.to_numpy()]
X_val_seq = X_seq[val_seq_mask.to_numpy()]
X_test_seq_all = X_seq[test_seq_mask.to_numpy()]

meta_train = seq_meta.loc[train_seq_mask].reset_index(drop=True)
meta_val = seq_meta.loc[val_seq_mask].reset_index(drop=True)
meta_test_all = seq_meta.loc[test_seq_mask].reset_index(drop=True)

y_train_return = meta_train["return_scaled"].to_numpy(dtype=np.float32)
y_train_alpha = meta_train["alpha_scaled"].to_numpy(dtype=np.float32)
y_train_vol = meta_train["vol_scaled"].to_numpy(dtype=np.float32)
y_train_tail = meta_train["tail_scaled"].to_numpy(dtype=np.float32)
y_train_downside = meta_train["target_bottom_quintile"].to_numpy(dtype=np.float32)
y_train_regime = meta_train["market_regime_code"].to_numpy(dtype=np.int64)
w_train = meta_train["sample_weight"].to_numpy(dtype=np.float32)

y_val_return = meta_val["return_scaled"].to_numpy(dtype=np.float32)
y_val_alpha = meta_val["alpha_scaled"].to_numpy(dtype=np.float32)
y_val_vol = meta_val["vol_scaled"].to_numpy(dtype=np.float32)
y_val_tail = meta_val["tail_scaled"].to_numpy(dtype=np.float32)
y_val_downside = meta_val["target_bottom_quintile"].to_numpy(dtype=np.float32)
y_val_regime = meta_val["market_regime_code"].to_numpy(dtype=np.int64)
w_val = meta_val["sample_weight"].to_numpy(dtype=np.float32)

print("All sequences:", X_seq.shape)
print("Train sequences:", X_train_seq.shape)
print("Val sequences:", X_val_seq.shape)
print("Test sequences:", X_test_seq_all.shape)

All sequences: (53113, 20, 60)
Train sequences: (32295, 20, 60)
Val sequences: (12625, 20, 60)
Test sequences: (6150, 20, 60)


## Volatility-Focused LSTM-AE

In [8]:
class VolatilityFocusedLSTMAutoencoder(nn.Module):
    def __init__(self, input_dim: int, hidden_size: int, latent_dim: int, num_layers: int, dropout: float, regime_classes: int):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_size = int(hidden_size)
        self.latent_dim = int(latent_dim)
        self.num_layers = int(num_layers)
        self.dropout = float(dropout)
        self.regime_classes = int(regime_classes)
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.to_latent = nn.Sequential(nn.Linear(hidden_size, latent_dim), nn.LayerNorm(latent_dim), nn.Tanh())
        self.decoder_seed = nn.Linear(latent_dim, hidden_size)
        self.decoder = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.reconstruction_head = nn.Linear(hidden_size, input_dim)
        self.return_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.alpha_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.vol_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.tail_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.downside_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.regime_head = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, regime_classes))

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        _, (h_n, _) = self.encoder(x)
        latent = self.to_latent(h_n[-1])
        repeated = self.decoder_seed(latent).unsqueeze(1).repeat(1, x.shape[1], 1)
        decoded, _ = self.decoder(repeated)
        return {
            "reconstruction": self.reconstruction_head(decoded),
            "expected_return": self.return_head(latent).squeeze(-1),
            "expected_alpha": self.alpha_head(latent).squeeze(-1),
            "expected_vol_log": self.vol_head(latent).squeeze(-1),
            "expected_tail_log": self.tail_head(latent).squeeze(-1),
            "downside_logit": self.downside_head(latent).squeeze(-1),
            "regime_logits": self.regime_head(latent),
            "latent": latent,
        }


def make_reconstruction_feature_weights(feature_names: list[str]) -> pd.Series:
    weights = pd.Series(1.0, index=feature_names, dtype=float)
    emphasis_terms = ["vol", "downside", "drawdown", "corr", "stress", "liquidity", "gap", "range", "beta", "breadth"]
    for name in feature_names:
        if any(term in name for term in emphasis_terms):
            weights.loc[name] = 2.0
    return weights / weights.mean()


reconstruction_feature_weights = make_reconstruction_feature_weights(ALL_FEATURE_NAMES)
reconstruction_weight_tensor = torch.as_tensor(reconstruction_feature_weights.to_numpy(dtype=np.float32)).view(1, 1, -1)


def apply_feature_denoising(x: torch.Tensor, mask_rate: float) -> torch.Tensor:
    if mask_rate <= 0.0:
        return x
    keep = (torch.rand_like(x) > mask_rate).to(x.dtype)
    return x * keep


def weighted_mse(pred: torch.Tensor, target: torch.Tensor, sample_weight: torch.Tensor) -> torch.Tensor:
    return (sample_weight * (pred - target).pow(2)).mean()


def pairwise_rank_loss(pred_alpha: torch.Tensor, true_alpha: torch.Tensor, date_codes: torch.Tensor) -> torch.Tensor:
    losses = []
    for date_code in torch.unique(date_codes):
        idx = torch.where(date_codes == date_code)[0]
        if idx.numel() < 2:
            continue
        pred = pred_alpha[idx]
        target = true_alpha[idx]
        target_diff = target[:, None] - target[None, :]
        useful_pairs = target_diff > 0
        if useful_pairs.any():
            pred_diff = pred[:, None] - pred[None, :]
            losses.append(F.softplus(-pred_diff[useful_pairs]).mean())
    if not losses:
        return pred_alpha.new_tensor(0.0)
    return torch.stack(losses).mean()


def batch_loss(
    model: nn.Module,
    xb: torch.Tensor,
    y_return: torch.Tensor,
    y_alpha: torch.Tensor,
    y_vol: torch.Tensor,
    y_tail: torch.Tensor,
    y_downside: torch.Tensor,
    y_regime: torch.Tensor,
    sample_weight: torch.Tensor,
    date_codes: torch.Tensor,
    *,
    denoise: bool,
) -> tuple[torch.Tensor, dict[str, float]]:
    clean_x = xb
    noisy_x = apply_feature_denoising(clean_x, MASK_RATE) if denoise else clean_x
    output = model(noisy_x)
    feature_weights = reconstruction_weight_tensor.to(clean_x.device)
    recon_loss = ((output["reconstruction"] - clean_x).pow(2) * feature_weights).mean()
    return_loss = weighted_mse(output["expected_return"], y_return, sample_weight)
    alpha_loss = weighted_mse(output["expected_alpha"], y_alpha, sample_weight)
    vol_loss = weighted_mse(output["expected_vol_log"], y_vol, sample_weight)
    tail_loss = weighted_mse(output["expected_tail_log"], y_tail, sample_weight)
    downside_loss = F.binary_cross_entropy_with_logits(output["downside_logit"], y_downside, weight=sample_weight)
    regime_loss = F.cross_entropy(output["regime_logits"], y_regime, reduction="none")
    regime_loss = (regime_loss * sample_weight).mean()
    rank_loss = pairwise_rank_loss(output["expected_alpha"], y_alpha, date_codes)
    total = (
        RECON_WEIGHT * recon_loss
        + RETURN_WEIGHT * return_loss
        + ALPHA_WEIGHT * alpha_loss
        + VOL_WEIGHT * vol_loss
        + TAIL_WEIGHT * tail_loss
        + DOWNSIDE_WEIGHT * downside_loss
        + REGIME_WEIGHT * regime_loss
        + RANK_WEIGHT * rank_loss
    )
    parts = {
        "loss": float(total.detach().cpu()),
        "recon": float(recon_loss.detach().cpu()),
        "return": float(return_loss.detach().cpu()),
        "alpha": float(alpha_loss.detach().cpu()),
        "vol": float(vol_loss.detach().cpu()),
        "tail": float(tail_loss.detach().cpu()),
        "downside": float(downside_loss.detach().cpu()),
        "regime": float(regime_loss.detach().cpu()),
        "rank": float(rank_loss.detach().cpu()),
    }
    return total, parts


def iter_date_batches(X, y_return, y_alpha, y_vol, y_tail, y_downside, y_regime, sample_weight, dates, *, dates_per_batch, shuffle, rng):
    date_series = pd.Series(pd.to_datetime(dates))
    unique_dates = np.array(pd.unique(date_series))
    if shuffle:
        rng.shuffle(unique_dates)
    for start in range(0, len(unique_dates), dates_per_batch):
        chosen_dates = unique_dates[start : start + dates_per_batch]
        idx = np.flatnonzero(np.isin(date_series.to_numpy(), chosen_dates))
        if shuffle:
            rng.shuffle(idx)
        date_codes = pd.factorize(date_series.iloc[idx])[0].astype(np.int64)
        yield X[idx], y_return[idx], y_alpha[idx], y_vol[idx], y_tail[idx], y_downside[idx], y_regime[idx], sample_weight[idx], date_codes

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VolatilityFocusedLSTMAutoencoder(
    input_dim=len(ALL_FEATURE_NAMES),
    hidden_size=HIDDEN_SIZE,
    latent_dim=LATENT_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    regime_classes=len(REGIME_CLASSES),
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
train_dates = meta_train["date"].to_numpy(dtype="datetime64[ns]")
val_dates = meta_val["date"].to_numpy(dtype="datetime64[ns]")


def evaluate_losses(model: nn.Module) -> dict[str, float]:
    model.eval()
    rng = np.random.default_rng(RANDOM_SEED)
    keys = ["loss", "recon", "return", "alpha", "vol", "tail", "downside", "regime", "rank"]
    totals = {key: 0.0 for key in keys}
    totals["n"] = 0.0
    with torch.no_grad():
        for xb_np, yr_np, ya_np, yv_np, yt_np, yd_np, yg_np, sw_np, dc_np in iter_date_batches(
            X_val_seq, y_val_return, y_val_alpha, y_val_vol, y_val_tail, y_val_downside, y_val_regime, w_val, val_dates,
            dates_per_batch=DATES_PER_BATCH, shuffle=False, rng=rng,
        ):
            xb = torch.as_tensor(xb_np, dtype=torch.float32, device=device)
            yr = torch.as_tensor(yr_np, dtype=torch.float32, device=device)
            ya = torch.as_tensor(ya_np, dtype=torch.float32, device=device)
            yv = torch.as_tensor(yv_np, dtype=torch.float32, device=device)
            yt = torch.as_tensor(yt_np, dtype=torch.float32, device=device)
            yd = torch.as_tensor(yd_np, dtype=torch.float32, device=device)
            yg = torch.as_tensor(yg_np, dtype=torch.long, device=device)
            sw = torch.as_tensor(sw_np, dtype=torch.float32, device=device)
            dc = torch.as_tensor(dc_np, dtype=torch.long, device=device)
            _, parts = batch_loss(model, xb, yr, ya, yv, yt, yd, yg, sw, dc, denoise=False)
            batch_n = float(len(xb_np))
            for key in keys:
                totals[key] += parts[key] * batch_n
            totals["n"] += batch_n
    return {key: totals[key] / max(totals["n"], 1.0) for key in keys}


history = []
best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
best_val_loss = float("inf")
patience_counter = 0
rng = np.random.default_rng(RANDOM_SEED)
keys = ["loss", "recon", "return", "alpha", "vol", "tail", "downside", "regime", "rank"]

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_totals = {key: 0.0 for key in keys}
    train_totals["n"] = 0.0
    for xb_np, yr_np, ya_np, yv_np, yt_np, yd_np, yg_np, sw_np, dc_np in iter_date_batches(
        X_train_seq, y_train_return, y_train_alpha, y_train_vol, y_train_tail, y_train_downside, y_train_regime, w_train, train_dates,
        dates_per_batch=DATES_PER_BATCH, shuffle=True, rng=rng,
    ):
        xb = torch.as_tensor(xb_np, dtype=torch.float32, device=device)
        yr = torch.as_tensor(yr_np, dtype=torch.float32, device=device)
        ya = torch.as_tensor(ya_np, dtype=torch.float32, device=device)
        yv = torch.as_tensor(yv_np, dtype=torch.float32, device=device)
        yt = torch.as_tensor(yt_np, dtype=torch.float32, device=device)
        yd = torch.as_tensor(yd_np, dtype=torch.float32, device=device)
        yg = torch.as_tensor(yg_np, dtype=torch.long, device=device)
        sw = torch.as_tensor(sw_np, dtype=torch.float32, device=device)
        dc = torch.as_tensor(dc_np, dtype=torch.long, device=device)

        optimizer.zero_grad(set_to_none=True)
        loss, parts = batch_loss(model, xb, yr, ya, yv, yt, yd, yg, sw, dc, denoise=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        batch_n = float(len(xb_np))
        for key in keys:
            train_totals[key] += parts[key] * batch_n
        train_totals["n"] += batch_n

    train_metrics = {key: train_totals[key] / max(train_totals["n"], 1.0) for key in keys}
    val_metrics = evaluate_losses(model)
    row = {"epoch": epoch}
    row.update({f"train_{k}": v for k, v in train_metrics.items()})
    row.update({f"val_{k}": v for k, v in val_metrics.items()})
    history.append(row)
    print(
        f"epoch {epoch:02d}/{EPOCHS} | train {train_metrics['loss']:.4f} | val {val_metrics['loss']:.4f} | "
        f"vol {val_metrics['vol']:.4f} | tail {val_metrics['tail']:.4f} | downside {val_metrics['downside']:.4f}"
    )

    if val_metrics["loss"] < best_val_loss - 1e-5:
        best_val_loss = val_metrics["loss"]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

model.load_state_dict(best_state)
history = pd.DataFrame(history)
display(history.tail())
print("Best validation loss:", best_val_loss)

epoch 01/18 | train 5.8750 | val 9.2785 | vol 2.0329 | tail 2.2684 | downside 0.7081
epoch 02/18 | train 5.2864 | val 9.0383 | vol 2.0798 | tail 2.2781 | downside 0.7076
epoch 03/18 | train 5.0084 | val 9.1518 | vol 2.1853 | tail 2.3158 | downside 0.7081
epoch 04/18 | train 4.8070 | val 9.3082 | vol 2.3272 | tail 2.4547 | downside 0.7086
epoch 05/18 | train 4.6052 | val 9.5729 | vol 2.4041 | tail 2.5423 | downside 0.7146
epoch 06/18 | train 4.4463 | val 9.8880 | vol 2.5331 | tail 2.6775 | downside 0.7206
epoch 07/18 | train 4.3020 | val 9.9931 | vol 2.6413 | tail 2.7518 | downside 0.7214
Early stopping at epoch 7.


,epoch,train_loss,train_recon,train_return,train_alpha,train_vol,train_tail,train_downside,train_regime,train_rank,val_loss,val_recon,val_return,val_alpha,val_vol,val_tail,val_downside,val_regime,val_rank
2,3,5.008448,0.602005,1.374894,1.309503,0.900032,1.131614,0.656700,0.775578,0.687760,9.151832,1.793482,2.257521,1.922306,2.185286,2.315821,0.708124,1.054987,0.698368
3,4,4.806968,0.572464,1.321522,1.270774,0.861903,1.076300,0.653355,0.669750,0.683659,9.308246,1.749658,2.298494,1.945527,2.327236,2.454723,0.708578,0.923607,0.700667
4,5,4.605208,0.554235,1.245000,1.206671,0.837089,1.020662,0.645125,0.605330,0.676141,9.572929,1.759946,2.400778,2.034733,2.404078,2.542267,0.714643,0.916764,0.712584
5,6,4.446268,0.541844,1.184330,1.159997,0.803080,0.971597,0.641213,0.571511,0.671430,9.887957,1.703238,2.487506,2.146970,2.533096,2.677483,0.720571,0.945124,0.726339
6,7,4.302021,0.529429,1.106309,1.096427,0.794221,0.936134,0.636059,0.559327,0.662824,9.993113,1.665106,2.470593,2.148198,2.641327,2.751755,0.721364,0.946802,0.721033


Best validation loss: 9.038313852442373


## Prediction Scores And Regime Diagnostics

In [10]:
def predict_scaled(model: nn.Module, X: np.ndarray, batch_size: int = 4096) -> pd.DataFrame:
    model.eval()
    returns, alphas, vols, tails, downside_probs, latent_norms, regime_probs = [], [], [], [], [], [], []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.as_tensor(X[start : start + batch_size], dtype=torch.float32, device=device)
            output = model(xb)
            returns.append(output["expected_return"].detach().cpu().numpy())
            alphas.append(output["expected_alpha"].detach().cpu().numpy())
            vols.append(output["expected_vol_log"].detach().cpu().numpy())
            tails.append(output["expected_tail_log"].detach().cpu().numpy())
            downside_probs.append(torch.sigmoid(output["downside_logit"]).detach().cpu().numpy())
            regime_probs.append(torch.softmax(output["regime_logits"], dim=1).detach().cpu().numpy())
            latent_norms.append(torch.linalg.norm(output["latent"], dim=1).detach().cpu().numpy())
    result = pd.DataFrame(
        {
            "return_scaled_pred": np.concatenate(returns),
            "alpha_scaled_pred": np.concatenate(alphas),
            "vol_scaled_pred": np.concatenate(vols),
            "tail_scaled_pred": np.concatenate(tails),
            "expected_downside_risk": np.concatenate(downside_probs),
            "latent_norm": np.concatenate(latent_norms),
        }
    )
    regime_array = np.concatenate(regime_probs)
    for idx, name in enumerate(REGIME_CLASSES):
        result[f"regime_prob_{name}"] = regime_array[:, idx]
    return result


def unscale_predictions(predicted: pd.DataFrame) -> pd.DataFrame:
    result = predicted.copy()
    result["expected_return"] = result["return_scaled_pred"] * target_stds["return"] + target_means["return"]
    result["expected_alpha"] = result["alpha_scaled_pred"] * target_stds["alpha"] + target_means["alpha"]
    vol_log = result["vol_scaled_pred"] * target_stds["vol_log"] + target_means["vol_log"]
    tail_log = result["tail_scaled_pred"] * target_stds["tail_log"] + target_means["tail_log"]
    result["expected_forward_volatility"] = np.expm1(vol_log).clip(lower=1e-4)
    result["expected_tail_loss"] = np.expm1(tail_log).clip(lower=0.0)
    result["regime_confidence"] = (
        result["regime_prob_calm_risk_on"]
        + 0.50 * result["regime_prob_rebound"]
        - 0.75 * result["regime_prob_high_vol"]
        - result["regime_prob_drawdown"]
    )
    return result


def rank_normalize_by_date(frame: pd.DataFrame, score_col: str, out_col: str) -> pd.DataFrame:
    result = frame.copy()
    pct_rank = result.groupby("date")[score_col].rank(method="average", pct=True)
    result[out_col] = 2.0 * pct_rank - 1.0
    return result


def add_scores(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["expected_volatility"] = np.clip(result["historical_vol_20d"].to_numpy(dtype=float), 1e-4, None)
    result["uncertainty"] = np.abs(result["expected_return"] - result["expected_alpha"])
    result["alpha_hist_vol_score"] = result["expected_alpha"] / result["expected_volatility"].clip(lower=1e-4)
    result["alpha_pred_vol_score"] = result["expected_alpha"] / result["expected_forward_volatility"].clip(lower=1e-4)
    result["vol_blended_score"] = (
        result["expected_alpha"]
        / (0.50 * result["expected_volatility"] + 0.50 * result["expected_forward_volatility"] + 1e-4)
        - DOWNSIDE_SCORE_PENALTY * result["expected_downside_risk"]
        - TAIL_SCORE_PENALTY * result["expected_tail_loss"]
        - UNCERTAINTY_SCORE_PENALTY * result["uncertainty"].abs()
        + REGIME_SCORE_BONUS * result["regime_confidence"]
    )
    result["strong_downside_score"] = (
        result["expected_alpha"]
        / (0.25 * result["expected_volatility"] + 0.75 * result["expected_forward_volatility"] + 1e-4)
        - 1.35 * DOWNSIDE_SCORE_PENALTY * result["expected_downside_risk"]
        - 1.50 * TAIL_SCORE_PENALTY * result["expected_tail_loss"]
        - 0.50 * result["regime_prob_drawdown"]
        - 0.25 * result["regime_prob_high_vol"]
    )
    for col in ["alpha_hist_vol_score", "alpha_pred_vol_score", "vol_blended_score", "strong_downside_score"]:
        result = rank_normalize_by_date(result, col, f"{col}_rank")
    return result


def daily_rank_ic(frame: pd.DataFrame, score_col: str, target_col: str) -> pd.DataFrame:
    rows = []
    for date_value, group in frame.groupby("date", sort=True):
        if group[score_col].nunique() < 2 or group[target_col].nunique() < 2:
            continue
        rows.append({"date": pd.Timestamp(date_value), "rank_ic": group[score_col].rank().corr(group[target_col].rank())})
    return pd.DataFrame(rows)


def diagnostics_by_regime(frame: pd.DataFrame, score_col: str) -> pd.DataFrame:
    rows = []
    for regime_name, group in frame.groupby("market_regime", sort=True):
        rank_ic = daily_rank_ic(group, score_col, "target_alpha")
        rows.append(
            {
                "market_regime": regime_name,
                "rows": len(group),
                "rank_ic": float(rank_ic["rank_ic"].mean()) if not rank_ic.empty else np.nan,
                "avg_forward_vol": float(group["target_forward_vol"].mean()),
                "avg_tail_loss": float(group["target_tail_loss"].mean()),
                "avg_downside_prob": float(group["expected_downside_risk"].mean()),
            }
        )
    return pd.DataFrame(rows)


val_pred_scaled = unscale_predictions(predict_scaled(model, X_val_seq))
val_diagnostics = add_scores(pd.concat([meta_val.copy(), val_pred_scaled], axis=1))
val_return_mse = float(np.mean((val_diagnostics["expected_return"] - val_diagnostics["target_return"]) ** 2))
val_alpha_mse = float(np.mean((val_diagnostics["expected_alpha"] - val_diagnostics["target_alpha"]) ** 2))
val_vol_mse = float(np.mean((val_diagnostics["expected_forward_volatility"] - val_diagnostics["target_forward_vol"]) ** 2))
val_tail_mse = float(np.mean((val_diagnostics["expected_tail_loss"] - val_diagnostics["target_tail_loss"]) ** 2))
val_downside_brier = float(np.mean((val_diagnostics["expected_downside_risk"] - val_diagnostics["target_bottom_quintile"]) ** 2))
val_regime_accuracy = float((val_diagnostics[[f"regime_prob_{name}" for name in REGIME_CLASSES]].to_numpy().argmax(axis=1) == val_diagnostics["market_regime_code"].to_numpy()).mean())

score_diagnostic_rows = []
for score_col in ["alpha_hist_vol_score", "alpha_pred_vol_score", "vol_blended_score", "strong_downside_score"]:
    rank_ic = daily_rank_ic(val_diagnostics, score_col, "target_alpha")
    vol_rank_ic = daily_rank_ic(val_diagnostics, score_col, "target_forward_vol")
    score_diagnostic_rows.append(
        {
            "score_col": score_col,
            "rank_ic_alpha": float(rank_ic["rank_ic"].mean()) if not rank_ic.empty else np.nan,
            "rank_ic_forward_vol": float(vol_rank_ic["rank_ic"].mean()) if not vol_rank_ic.empty else np.nan,
            "avg_selected_downside_proxy": float(val_diagnostics.groupby("date").apply(lambda g: g.nlargest(max(1, len(g)//5), score_col)["target_tail_loss"].mean()).mean()),
        }
    )
score_validation_table = pd.DataFrame(score_diagnostic_rows)

print("Validation MSE/Brier:", {"return": val_return_mse, "alpha": val_alpha_mse, "vol": val_vol_mse, "tail": val_tail_mse, "downside": val_downside_brier, "regime_acc": val_regime_accuracy})
display(score_validation_table)
print("Regime diagnostics for strong downside score:")
display(diagnostics_by_regime(val_diagnostics, "strong_downside_score"))

Validation MSE/Brier: {'return': 0.0031348791543636466, 'alpha': 0.0020022614480077667, 'vol': 0.05228431747745003, 'tail': 0.0005481657263545331, 'downside': 0.15860061040471427, 'regime_acc': 0.6596435643564357}


,score_col,rank_ic_alpha,rank_ic_forward_vol,avg_selected_downside_proxy
0,alpha_hist_vol_score,-0.001081,-0.012710,0.026529
1,alpha_pred_vol_score,0.004687,0.006611,0.026553
2,vol_blended_score,-0.007383,-0.331185,0.021487
3,strong_downside_score,-0.007970,-0.339024,0.021421


Regime diagnostics for strong downside score:


,market_regime,rows,rank_ic,avg_forward_vol,avg_tail_loss,avg_downside_prob
0,calm_risk_on,4500,0.022893,0.246687,0.020991,0.190529
1,drawdown,3850,-0.030445,0.430204,0.032909,0.200809
2,high_vol,3300,-0.032372,0.278952,0.022541,0.196473
3,rebound,975,0.020927,0.374431,0.029818,0.257881


## Volatility-Controlled Portfolio Construction

In [11]:
def select_rebalance_dates(dates: pd.Series | pd.DatetimeIndex, frequency: str = "weekly") -> pd.DatetimeIndex:
    unique_dates = pd.DatetimeIndex(pd.to_datetime(pd.Series(dates).drop_duplicates()).sort_values())
    if frequency == "daily":
        return unique_dates
    date_series = pd.Series(unique_dates, index=unique_dates)
    if frequency == "weekly":
        return pd.DatetimeIndex(date_series.groupby(date_series.index.to_period("W-FRI")).max().to_numpy())
    if frequency == "monthly":
        return pd.DatetimeIndex(date_series.groupby(date_series.index.to_period("M")).max().to_numpy())
    if frequency == "every_5_trading_days":
        return unique_dates[::5]
    if frequency == "every_10_trading_days":
        return unique_dates[::10]
    raise ValueError("frequency must be one of: daily, weekly, monthly, every_5_trading_days, every_10_trading_days")


def apply_weight_cap(raw_weights: pd.Series, max_weight: float) -> pd.Series:
    raw = raw_weights.clip(lower=0.0).astype(float)
    raw = raw.loc[raw > 0.0]
    if raw.empty:
        return raw_weights * 0.0
    feasible_cap = max(float(max_weight), 1.0 / len(raw) + 1e-8)
    weights = pd.Series(0.0, index=raw.index, dtype=float)
    remaining = raw.copy()
    budget = 1.0
    while not remaining.empty:
        scaled = remaining / remaining.sum() * budget
        over_cap = scaled > feasible_cap
        if not over_cap.any():
            weights.loc[remaining.index] = scaled
            break
        capped_names = scaled.loc[over_cap].index
        weights.loc[capped_names] = feasible_cap
        budget -= feasible_cap * len(capped_names)
        remaining = remaining.drop(index=capped_names)
        if budget <= 1e-12:
            break
    total = weights.sum()
    if total <= 0.0:
        weights = pd.Series(1.0 / len(raw), index=raw.index, dtype=float)
    elif abs(total - 1.0) > 1e-10:
        weights = weights / total
    return weights


def _price_wide(prices: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    wide = prices.copy()
    wide["date"] = pd.to_datetime(wide["date"], utc=True).dt.tz_localize(None)
    wide["ticker"] = wide["ticker"].astype(str).str.upper()
    return wide.pivot(index="date", columns="ticker", values="adj_close").sort_index().reindex(columns=tickers)


price_wide = _price_wide(prices, tradable_tickers)
returns_wide = price_wide.pct_change()


def effective_name_count(weights: pd.Series) -> float:
    values = weights.fillna(0.0).to_numpy(dtype=float)
    denom = float(np.square(values).sum())
    return 1.0 / denom if denom > 0 else 0.0


def min_variance_anchor(date_value: pd.Timestamp, tickers: list[str], fallback_vol: pd.Series, max_weight: float) -> pd.Series:
    available_tickers = [ticker for ticker in tickers if ticker in returns_wide.columns]
    if not available_tickers:
        return pd.Series(1.0 / len(tickers), index=tickers, dtype=float)
    hist = returns_wide.loc[:pd.Timestamp(date_value), available_tickers].tail(COV_LOOKBACK_DAYS).dropna(how="all")
    if len(hist) < 40 or len(available_tickers) < 2:
        inv = 1.0 / fallback_vol.reindex(tickers).replace(0.0, np.nan).fillna(fallback_vol.median()).clip(lower=1e-4)
        return apply_weight_cap(inv, max_weight=max_weight).reindex(tickers).fillna(0.0)
    clean = hist.fillna(0.0)
    try:
        cov = LedoitWolf().fit(clean.to_numpy(dtype=float)).covariance_
        inv_cov = np.linalg.pinv(cov + np.eye(cov.shape[0]) * 1e-8)
        ones = np.ones(len(available_tickers))
        raw = inv_cov @ ones
        raw = np.clip(raw, 0.0, None)
        if raw.sum() <= 0.0:
            raw = 1.0 / np.diag(cov).clip(min=1e-8)
        anchor = pd.Series(raw, index=available_tickers, dtype=float)
    except Exception:
        inv = 1.0 / fallback_vol.reindex(available_tickers).replace(0.0, np.nan).fillna(fallback_vol.median()).clip(lower=1e-4)
        anchor = inv.astype(float)
    anchor = apply_weight_cap(anchor, max_weight=max_weight)
    return anchor.reindex(tickers).fillna(0.0)


def _is_unfavorable_regime(group: pd.DataFrame) -> bool:
    return bool(
        (float(group["spy_vol_20d_ann"].median()) >= regime_thresholds["high_vol_spy_vol_20d_ann"])
        or (float(group["spy_drawdown_60d"].median()) <= regime_thresholds["drawdown_spy_drawdown_60d"])
        or (float(group["benchmark_agnostic_stress"].median()) >= regime_thresholds["stress_benchmark_agnostic"])
        or (float(group["regime_prob_drawdown"].mean()) >= 0.28)
        or (float(group["regime_prob_high_vol"].mean()) >= 0.35)
    )


def weights_from_volatility_controlled_scores(
    predictions: pd.DataFrame,
    *,
    score_col: str,
    dataset_name,
    strategy_name: str,
    frequency: str,
    high_vol_max_weight: float,
    normal_max_weight: float,
    turnover_blend: float,
    beta_target: float,
    universe_tickers: list[str],
) -> tuple[PortfolioWeights, pd.DataFrame]:
    tickers = [ticker for ticker in universe_tickers if ticker in set(predictions["ticker"])]
    rows = []
    logs = []
    previous = None
    previous_date = None
    proxy_nav = 1.0
    proxy_peak = 1.0

    for date_value, group in predictions.groupby("date", sort=True):
        date_value = pd.Timestamp(date_value)
        if previous is not None and previous_date is not None and previous_date in price_wide.index and date_value in price_wide.index:
            period_returns = price_wide.loc[date_value, tickers] / price_wide.loc[previous_date, tickers] - 1.0
            period_returns = period_returns.replace([np.inf, -np.inf], np.nan).fillna(0.0)
            proxy_nav *= float(1.0 + (previous * period_returns).sum())
            proxy_peak = max(proxy_peak, proxy_nav)
        proxy_drawdown = proxy_nav / max(proxy_peak, 1e-12) - 1.0

        unfavorable = _is_unfavorable_regime(group)
        effective_max_weight = high_vol_max_weight if unfavorable else normal_max_weight
        if proxy_drawdown <= -0.08:
            effective_max_weight = min(effective_max_weight, 0.05)
        if proxy_drawdown <= -0.15:
            effective_max_weight = min(effective_max_weight, 0.03)
        min_names = HIGH_RISK_MIN_ACTIVE_NAMES if unfavorable else NORMAL_MIN_ACTIVE_NAMES
        min_names = min(len(tickers), max(min_names, int(math.ceil(1.0 / effective_max_weight))))
        target_fraction = 0.90 if unfavorable else BASE_TOP_FRACTION

        group = group.sort_values(score_col, ascending=False).reset_index(drop=True)
        top_n = min(len(group), max(min_names, int(math.ceil(len(group) * target_fraction))))
        selected = group.head(top_n).copy()
        selected_tickers = [ticker for ticker in selected["ticker"] if ticker in tickers]
        fallback_vol = selected.set_index("ticker")["expected_forward_volatility"].reindex(selected_tickers).fillna(selected["expected_forward_volatility"].median())
        anchor = min_variance_anchor(date_value, selected_tickers, fallback_vol, max_weight=effective_max_weight)

        raw_signal = selected.set_index("ticker")[score_col].reindex(selected_tickers).astype(float)
        if score_col.endswith("_rank"):
            tilt_raw = (raw_signal - raw_signal.min() + 1e-6)
        else:
            tilt_raw = raw_signal.rank(pct=True).clip(lower=0.0)
        risk_divisor = (
            selected.set_index("ticker")["expected_forward_volatility"].reindex(selected_tickers).clip(lower=1e-4)
            * (1.0 + selected.set_index("ticker")["expected_downside_risk"].reindex(selected_tickers).fillna(0.5))
            * (1.0 + 10.0 * selected.set_index("ticker")["expected_tail_loss"].reindex(selected_tickers).fillna(0.0))
        )
        tilt_raw = (tilt_raw / risk_divisor).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        tilt = apply_weight_cap(tilt_raw, max_weight=effective_max_weight).reindex(selected_tickers).fillna(0.0)
        confidence_ok = (
            float(selected["expected_downside_risk"].mean()) < 0.35
            and float(selected["regime_confidence"].mean()) > -0.15
            and float(selected["uncertainty"].mean()) < selected["uncertainty"].quantile(0.75)
        )
        tilt_strength = MINVAR_TILT_HIGH_RISK if unfavorable else MINVAR_TILT_NORMAL
        if not confidence_ok:
            tilt_strength *= 0.50
        if proxy_drawdown <= -0.08:
            tilt_strength *= 0.50

        combined = (1.0 - tilt_strength) * anchor + tilt_strength * tilt
        combined = apply_weight_cap(combined, max_weight=effective_max_weight)
        row = pd.Series(0.0, index=tickers, dtype=float)
        row.loc[combined.index] = combined.to_numpy(dtype=float)
        row = row / row.sum()

        betas = group.drop_duplicates("ticker").set_index("ticker").reindex(tickers)["beta_60d_spy"].astype(float).fillna(1.0)
        portfolio_beta = float((row * betas).sum())
        beta_blend = 0.0
        if portfolio_beta > beta_target:
            inv_vol = 1.0 / group.drop_duplicates("ticker").set_index("ticker").reindex(tickers)["expected_forward_volatility"].astype(float).clip(lower=1e-4)
            inv_vol = apply_weight_cap(inv_vol.fillna(inv_vol.median()), max_weight=effective_max_weight).reindex(tickers).fillna(0.0)
            inv_beta = float((inv_vol * betas).sum())
            if portfolio_beta > inv_beta:
                beta_blend = float(np.clip((portfolio_beta - beta_target) / max(portfolio_beta - inv_beta, 1e-6), 0.0, 0.85))
                row = (1.0 - beta_blend) * row + beta_blend * inv_vol
                row = row / row.sum()
                portfolio_beta = float((row * betas).sum())

        pre_turnover_row = row.copy()
        if previous is not None:
            row = turnover_blend * row + (1.0 - turnover_blend) * previous
            row = row / row.sum()
        turnover_estimate = float((row.fillna(0.0) - (previous.fillna(0.0) if previous is not None else 0.0)).abs().sum() / 2.0)
        row.name = date_value
        rows.append(row)
        logs.append(
            {
                "date": date_value,
                "score_col": score_col,
                "frequency": frequency,
                "unfavorable_regime": unfavorable,
                "effective_max_weight": effective_max_weight,
                "selected_names": len(selected_tickers),
                "active_names": int((row > 1e-8).sum()),
                "effective_names": effective_name_count(row),
                "realized_max_weight": float(row.max()),
                "portfolio_beta_estimate": portfolio_beta,
                "beta_blend": beta_blend,
                "tilt_strength": tilt_strength,
                "turnover_estimate": turnover_estimate,
                "proxy_drawdown": proxy_drawdown,
            }
        )
        previous = row
        previous_date = date_value

    weights = pd.DataFrame(rows)
    weights.index.name = "date"
    metadata = {
        "score_col": score_col,
        "rebalance_frequency": frequency,
        "normal_max_weight": normal_max_weight,
        "high_vol_max_weight": high_vol_max_weight,
        "turnover_blend": turnover_blend,
        "beta_target": beta_target,
        "minvar_anchor": True,
        "universe_ticker_count": len(tickers),
    }
    return PortfolioWeights(weights=weights, dataset_name=dataset_name.identifier, strategy_name=strategy_name, metadata=metadata), pd.DataFrame(logs)

In [12]:
test_pred_scaled = unscale_predictions(predict_scaled(model, X_test_seq_all))
test_scored_all = add_scores(pd.concat([meta_test_all.copy(), test_pred_scaled], axis=1))

score_columns = ["alpha_hist_vol_score", "alpha_pred_vol_score", "vol_blended_score", "strong_downside_score"]
if FAST_SMOKE:
    max_weight_grid = [0.05]
    score_grid = ["strong_downside_score"]
    frequency_grid = ["weekly"]
else:
    max_weight_grid = HIGH_VOL_MAX_WEIGHT_OPTIONS
    score_grid = score_columns
    frequency_grid = ["weekly", "every_5_trading_days", "every_10_trading_days"]

candidate_portfolios = {}
candidate_logs = {}
candidate_predictions = {}
for frequency in frequency_grid:
    rebalance_dates = select_rebalance_dates(test_scored_all["date"], frequency)
    scored = test_scored_all.loc[test_scored_all["date"].isin(rebalance_dates)].copy()
    for score_col in score_grid:
        for high_vol_max_weight in max_weight_grid:
            name = f"{frequency}_{score_col}_hvw{str(high_vol_max_weight).replace('.', '')}"
            prediction_frame = scored.loc[:, [
                "date", "ticker", "expected_return", "expected_alpha", "expected_volatility", "expected_forward_volatility",
                "expected_tail_loss", "expected_downside_risk", "uncertainty", "regime_confidence",
                "regime_prob_calm_risk_on", "regime_prob_high_vol", "regime_prob_drawdown", "regime_prob_rebound",
                "spy_vol_20d_ann", "spy_drawdown_60d", "spy_momentum_20d", "beta_60d_spy", "benchmark_agnostic_stress",
                *score_columns, *[f"{col}_rank" for col in score_columns],
            ]].copy()
            prediction_frame["horizon"] = HORIZON
            prediction_frame = prediction_frame.loc[:, ["date", "ticker", "horizon"] + [c for c in prediction_frame.columns if c not in {"date", "ticker", "horizon"}]]
            prediction_frame = validate_prediction_frame(prediction_frame, dataset_name=research_spec, horizon=HORIZON, repo_root=repo_root)
            rank_score_col = f"{score_col}_rank"
            portfolio_obj, log_frame = weights_from_volatility_controlled_scores(
                prediction_frame,
                score_col=rank_score_col,
                dataset_name=research_spec,
                strategy_name=f"{MODEL_NAME}_{name}",
                frequency=frequency,
                high_vol_max_weight=high_vol_max_weight,
                normal_max_weight=NORMAL_MAX_WEIGHT,
                turnover_blend=TURNOVER_BLEND,
                beta_target=BETA_TARGET,
                universe_tickers=tradable_tickers,
            )
            candidate_portfolios[name] = portfolio_obj
            candidate_logs[name] = log_frame
            candidate_predictions[name] = prediction_frame

print("Candidate count:", len(candidate_portfolios))
first_name = next(iter(candidate_portfolios))
display(candidate_predictions[first_name].head())
display(candidate_logs[first_name].head())

Candidate count: 48


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,expected_forward_volatility,expected_tail_loss,expected_downside_risk,uncertainty,...,beta_60d_spy,benchmark_agnostic_stress,alpha_hist_vol_score,alpha_pred_vol_score,vol_blended_score,strong_downside_score,alpha_hist_vol_score_rank,alpha_pred_vol_score_rank,vol_blended_score_rank,strong_downside_score_rank
0,2022-01-07,AAPL,5,0.005275,0.004988,0.291345,0.219727,0.020037,0.152104,0.000288,...,1.164415,0.765577,0.017119,0.022699,-0.188023,-0.604908,-0.44,-0.28,0.60,0.36
1,2022-01-07,ADBE,5,0.010866,0.009812,0.519157,0.305341,0.024806,0.258852,0.001054,...,2.093541,0.765577,0.018901,0.032136,-0.264365,-0.702027,-0.20,0.20,-0.68,-0.84
2,2022-01-07,ADI,5,0.010983,0.010342,0.242832,0.245846,0.017838,0.175376,0.000641,...,1.297955,0.765577,0.042590,0.042068,-0.178156,-0.590746,0.68,0.44,0.76,0.60
3,2022-01-07,AMAT,5,0.011979,0.011773,0.408458,0.378215,0.030387,0.185531,0.000207,...,1.853227,0.765577,0.028822,0.031127,-0.217259,-0.660968,0.28,0.12,-0.20,-0.28
4,2022-01-07,AMD,5,0.012489,0.016148,0.588302,0.435677,0.034362,0.241010,0.003659,...,2.407290,0.765577,0.027448,0.037064,-0.256848,-0.693786,0.20,0.36,-0.60,-0.60


,date,score_col,frequency,unfavorable_regime,effective_max_weight,selected_names,active_names,effective_names,realized_max_weight,portfolio_beta_estimate,beta_blend,tilt_strength,turnover_estimate,proxy_drawdown
0,2022-01-07,alpha_hist_vol_score_rank,weekly,True,0.03,25,25,25.0,0.04,1.552897,0.85,0.050,5.000000e-01,0.000000
1,2022-01-14,alpha_hist_vol_score_rank,weekly,True,0.03,25,25,25.0,0.04,1.606292,0.85,0.050,9.562500e-08,0.000000
2,2022-01-21,alpha_hist_vol_score_rank,weekly,True,0.03,25,25,25.0,0.04,1.630897,0.00,0.025,6.018750e-08,-0.097957
3,2022-01-28,alpha_hist_vol_score_rank,weekly,True,0.03,25,25,25.0,0.04,1.610201,0.85,0.025,1.114559e-07,-0.103501
4,2022-02-04,alpha_hist_vol_score_rank,weekly,True,0.03,25,25,25.0,0.04,1.668309,0.85,0.050,6.889447e-08,-0.079555


## Backtests And Volatile-Regime Stress Tables

In [13]:
def write_artifacts_for_notebook(result, output_dir: Path) -> dict[str, str]:
    if os.environ.get("SKIP_QUANTSTATS", "0") != "1":
        return write_backtest_artifacts(result, output_dir)
    output_path = Path(output_dir).resolve()
    output_path.mkdir(parents=True, exist_ok=True)
    paths = {
        "weights": output_path / "weights.parquet",
        "nav": output_path / "nav.parquet",
        "returns": output_path / "returns.parquet",
        "turnover": output_path / "turnover.parquet",
        "benchmarks": output_path / "benchmarks.parquet",
        "metrics": output_path / "metrics.json",
        "metrics_table": output_path / "metrics_table.parquet",
        "quantstats_report": output_path / "quantstats.html",
    }
    result.weights.to_parquet(paths["weights"])
    result.nav.to_frame(name="nav").to_parquet(paths["nav"])
    result.returns.to_frame(name="returns").to_parquet(paths["returns"])
    result.turnover.to_frame(name="turnover").to_parquet(paths["turnover"])
    result.benchmark_returns.to_parquet(paths["benchmarks"])
    paths["metrics"].write_text(json.dumps(result.metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    pd.DataFrame([{"metric": key, "value": value} for key, value in sorted(result.metrics.items())]).to_parquet(paths["metrics_table"], index=False)
    paths["quantstats_report"].write_text("<html><body><p>QuantStats skipped because SKIP_QUANTSTATS=1.</p></body></html>\n", encoding="utf-8")
    artifact_paths = {key: str(value) for key, value in paths.items()}
    result.artifact_paths.update(artifact_paths)
    return artifact_paths


def stress_table(result, benchmark_name: str) -> pd.DataFrame:
    returns = result.returns.astype(float)
    benchmark_returns = result.benchmark_returns.iloc[:, 0].astype(float)
    if benchmark_name in result.benchmark_returns.columns:
        benchmark_returns = result.benchmark_returns[benchmark_name].astype(float)
    common = pd.concat([returns.rename("strategy"), benchmark_returns.rename("benchmark")], axis=1).dropna()
    if not result.weights.empty:
        common = common.loc[(common.index >= result.weights.index.min()) & (common.index <= result.weights.index.max())]
    rolling_5d = (1.0 + common["strategy"]).rolling(5).apply(np.prod, raw=True) - 1.0
    monthly = (1.0 + common["strategy"]).resample("M").prod() - 1.0
    crash_threshold = common["benchmark"].quantile(0.10)
    rebound_threshold = common["benchmark"].quantile(0.90)
    beta = common["strategy"].cov(common["benchmark"]) / max(common["benchmark"].var(), 1e-12)
    return pd.DataFrame(
        [
            {"stress_metric": "realized_volatility", "value": float(common["strategy"].std(ddof=0) * np.sqrt(252.0))},
            {"stress_metric": "beta_to_benchmark", "value": float(beta)},
            {"stress_metric": "worst_day", "value": float(common["strategy"].min())},
            {"stress_metric": "worst_5_day_period", "value": float(rolling_5d.min())},
            {"stress_metric": "worst_month", "value": float(monthly.min())},
            {"stress_metric": "avg_return_benchmark_worst_decile_days", "value": float(common.loc[common["benchmark"] <= crash_threshold, "strategy"].mean())},
            {"stress_metric": "avg_return_benchmark_rebound_days", "value": float(common.loc[common["benchmark"] >= rebound_threshold, "strategy"].mean())},
            {"stress_metric": "benchmark_worst_decile_threshold", "value": float(crash_threshold)},
        ]
    )


def regime_return_table(result, context_frame: pd.DataFrame, benchmark_name: str) -> pd.DataFrame:
    returns = result.returns.rename("strategy_return").to_frame()
    benchmark_returns = result.benchmark_returns.iloc[:, 0].rename("benchmark_return")
    if benchmark_name in result.benchmark_returns.columns:
        benchmark_returns = result.benchmark_returns[benchmark_name].rename("benchmark_return")
    context = context_frame.loc[:, ["date", "spy_vol_20d_ann", "spy_drawdown_60d", "market_regime", "benchmark_agnostic_stress"]].drop_duplicates("date").copy()
    context["date"] = pd.to_datetime(context["date"])
    frame = returns.join(benchmark_returns, how="left").reset_index()
    frame = frame.rename(columns={frame.columns[0]: "date"})
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.merge(context, on="date", how="left").dropna(subset=["strategy_return", "benchmark_return"])
    if not result.weights.empty:
        frame = frame.loc[(frame["date"] >= result.weights.index.min()) & (frame["date"] <= result.weights.index.max())]
    crash_threshold = frame["benchmark_return"].quantile(0.10)
    rebound_threshold = frame["benchmark_return"].quantile(0.90)
    masks = {
        "benchmark_up_days": frame["benchmark_return"] > 0,
        "benchmark_down_days": frame["benchmark_return"] < 0,
        "high_benchmark_vol": frame["spy_vol_20d_ann"] >= regime_thresholds["high_vol_spy_vol_20d_ann"],
        "benchmark_worst_decile_days": frame["benchmark_return"] <= crash_threshold,
        "benchmark_rebound_days": frame["benchmark_return"] >= rebound_threshold,
        "market_drawdown_regime": frame["market_regime"].eq("drawdown"),
        "market_high_vol_regime": frame["market_regime"].eq("high_vol"),
        "benchmark_agnostic_stress": frame["benchmark_agnostic_stress"] >= regime_thresholds["stress_benchmark_agnostic"],
    }
    rows = []
    for name, mask in masks.items():
        subset = frame.loc[mask]
        rows.append(
            {
                "bucket": name,
                "observations": int(len(subset)),
                "avg_strategy_return": float(subset["strategy_return"].mean()) if len(subset) else np.nan,
                "strategy_vol_ann": float(subset["strategy_return"].std(ddof=0) * np.sqrt(252.0)) if len(subset) else np.nan,
                "avg_benchmark_return": float(subset["benchmark_return"].mean()) if len(subset) else np.nan,
                "hit_rate": float((subset["strategy_return"] > 0).mean()) if len(subset) else np.nan,
            }
        )
    return pd.DataFrame(rows)


candidate_results = {}
comparison_rows = []
for name, candidate in candidate_portfolios.items():
    print("Running backtest:", name)
    candidate_result = backtest_weights(backtest_spec, candidate, benchmark=research_spec.benchmark_ticker, repo_root=repo_root)
    candidate_result.metrics = build_metrics(candidate_result)
    candidate_results[name] = candidate_result
    stress = stress_table(candidate_result, research_spec.benchmark_ticker)
    stress_map = dict(zip(stress["stress_metric"], stress["value"]))
    log_frame = candidate_logs[name]
    row = {"candidate": name}
    for metric in ["annual_return", "annual_volatility", "sharpe", "sortino", "max_drawdown", "average_turnover", "total_return"]:
        row[metric] = candidate_result.metrics.get(metric, np.nan)
    row.update(
        {
            "stress_worst_day": stress_map.get("worst_day"),
            "stress_worst_5_day": stress_map.get("worst_5_day_period"),
            "stress_crash_day_avg": stress_map.get("avg_return_benchmark_worst_decile_days"),
            "beta_to_benchmark": stress_map.get("beta_to_benchmark"),
            "avg_active_names": float((candidate_result.weights > 1e-8).sum(axis=1).mean()),
            "avg_effective_names": float(log_frame["effective_names"].mean()) if not log_frame.empty else np.nan,
            "avg_max_weight": float(log_frame["realized_max_weight"].mean()) if not log_frame.empty else np.nan,
            "avg_beta_estimate": float(log_frame["portfolio_beta_estimate"].mean()) if not log_frame.empty else np.nan,
        }
    )
    comparison_rows.append(row)

backtest_comparison = pd.DataFrame(comparison_rows)
backtest_comparison["risk_selection_score"] = (
    backtest_comparison["annual_volatility"]
    + 0.50 * backtest_comparison["max_drawdown"].abs()
    + 2.00 * backtest_comparison["stress_worst_day"].abs()
    + 1.50 * backtest_comparison["stress_crash_day_avg"].abs()
    + 0.05 * backtest_comparison["average_turnover"].fillna(0.0)
)
backtest_comparison = backtest_comparison.sort_values(["risk_selection_score", "annual_volatility", "max_drawdown"]).reset_index(drop=True)
selected_candidate_name = str(backtest_comparison.iloc[0]["candidate"])
result = candidate_results[selected_candidate_name]
portfolio = candidate_portfolios[selected_candidate_name]
risk_log = candidate_logs[selected_candidate_name]
predictions = candidate_predictions[selected_candidate_name]
validated_weights = validate_weights_frame(portfolio.weights, dataset_name=research_spec, repo_root=repo_root)
metrics = result.metrics
artifact_paths = write_artifacts_for_notebook(result, output_dir)
final_stress_table = stress_table(result, research_spec.benchmark_ticker)
final_regime_return_table = regime_return_table(result, feature_frame, research_spec.benchmark_ticker)

backtest_comparison.to_parquet(output_dir / "volatility_candidate_comparison.parquet", index=False)
final_stress_table.to_parquet(output_dir / "volatility_stress_table.parquet", index=False)
final_regime_return_table.to_parquet(output_dir / "volatility_regime_return_table.parquet", index=False)
risk_log.to_parquet(output_dir / "volatility_risk_log.parquet", index=False)
score_validation_table.to_parquet(output_dir / "volatility_score_validation_table.parquet", index=False)

print("Selected candidate:", selected_candidate_name)
display(backtest_comparison.head(12))
print("Final stress table:")
display(final_stress_table)
print("Regime return table:")
display(final_regime_return_table)

Running backtest: weekly_alpha_hist_vol_score_hvw003
Running backtest: weekly_alpha_hist_vol_score_hvw005
Running backtest: weekly_alpha_hist_vol_score_hvw0075
Running backtest: weekly_alpha_hist_vol_score_hvw01
Running backtest: weekly_alpha_pred_vol_score_hvw003
Running backtest: weekly_alpha_pred_vol_score_hvw005
Running backtest: weekly_alpha_pred_vol_score_hvw0075
Running backtest: weekly_alpha_pred_vol_score_hvw01
Running backtest: weekly_vol_blended_score_hvw003
Running backtest: weekly_vol_blended_score_hvw005
Running backtest: weekly_vol_blended_score_hvw0075
Running backtest: weekly_vol_blended_score_hvw01
Running backtest: weekly_strong_downside_score_hvw003
Running backtest: weekly_strong_downside_score_hvw005
Running backtest: weekly_strong_downside_score_hvw0075
Running backtest: weekly_strong_downside_score_hvw01
Running backtest: every_5_trading_days_alpha_hist_vol_score_hvw003
Running backtest: every_5_trading_days_alpha_hist_vol_score_hvw005
Running backtest: every_5_

Selected candidate: weekly_vol_blended_score_hvw01


,candidate,annual_return,annual_volatility,sharpe,sortino,max_drawdown,average_turnover,total_return,stress_worst_day,stress_worst_5_day,stress_crash_day_avg,beta_to_benchmark,avg_active_names,avg_effective_names,avg_max_weight,avg_beta_estimate,risk_selection_score
0,weekly_vol_blended_score_hvw01,-0.303695,0.371610,-0.817240,-1.420967,-0.366763,0.049686,-0.297978,-0.058752,-0.120396,-0.040607,1.442660,24.980392,22.871519,0.050452,1.467084,0.735890
1,weekly_strong_downside_score_hvw01,-0.305610,0.372453,-0.820534,-1.432250,-0.368505,0.048254,-0.299866,-0.058756,-0.120399,-0.040616,1.445052,25.000000,23.055079,0.049461,1.471807,0.737554
2,weekly_alpha_hist_vol_score_hvw01,-0.304369,0.373295,-0.815359,-1.422524,-0.367457,0.049150,-0.298643,-0.058842,-0.120406,-0.040694,1.448770,24.980392,22.952271,0.050296,1.473352,0.738204
3,weekly_alpha_pred_vol_score_hvw01,-0.305271,0.373370,-0.817609,-1.426821,-0.368243,0.048964,-0.299532,-0.058824,-0.120405,-0.040706,1.449061,24.980392,22.962920,0.050280,1.473933,0.738647
4,weekly_vol_blended_score_hvw0075,-0.303796,0.374246,-0.811756,-1.421545,-0.368698,0.046558,-0.298078,-0.058752,-0.120404,-0.040599,1.451163,24.980392,23.413452,0.046231,1.481364,0.739325
5,weekly_strong_downside_score_hvw0075,-0.304444,0.374387,-0.813180,-1.423797,-0.369331,0.046549,-0.298717,-0.058756,-0.120404,-0.040602,1.451374,25.000000,23.418964,0.046231,1.481587,0.739794
6,weekly_alpha_hist_vol_score_hvw0075,-0.304648,0.375124,-0.812125,-1.422642,-0.369560,0.047540,-0.298918,-0.058842,-0.120411,-0.040678,1.454458,24.980392,23.332294,0.046692,1.483389,0.740982
7,weekly_alpha_pred_vol_score_hvw0075,-0.305624,0.375212,-0.814538,-1.427095,-0.370390,0.047295,-0.299880,-0.058824,-0.120410,-0.040695,1.454770,24.980392,23.338093,0.046682,1.483932,0.741462
8,weekly_vol_blended_score_hvw005,-0.312255,0.377368,-0.827456,-1.460548,-0.376506,0.030361,-0.306415,-0.058901,-0.120412,-0.040594,1.461897,25.000000,24.290956,0.042985,1.499300,0.745833
9,weekly_strong_downside_score_hvw005,-0.312811,0.377502,-0.828635,-1.463310,-0.377710,0.030903,-0.306963,-0.058901,-0.120412,-0.040598,1.461948,25.000000,24.293408,0.042985,1.499236,0.746602


Final stress table:


,stress_metric,value
0,realized_volatility,0.372830
1,beta_to_benchmark,1.442660
2,worst_day,-0.058752
3,worst_5_day_period,-0.120396
4,worst_month,-0.141508
5,avg_return_benchmark_worst_decile_days,-0.040607
6,avg_return_benchmark_rebound_days,0.036878
7,benchmark_worst_decile_threshold,-0.019467


Regime return table:


,bucket,observations,avg_strategy_return,strategy_vol_ann,avg_benchmark_return,hit_rate
0,benchmark_up_days,106,0.019154,0.242987,0.013114,0.915094
1,benchmark_down_days,135,-0.017157,0.237987,-0.011471,0.125926
2,high_benchmark_vol,239,-0.001082,0.373738,-0.000581,0.472803
3,benchmark_worst_decile_days,25,-0.040607,0.144818,-0.027963,0.000000
4,benchmark_rebound_days,25,0.036878,0.221656,0.026290,1.000000
5,market_drawdown_regime,158,-0.004028,0.387493,-0.002619,0.449367
6,market_high_vol_regime,39,0.001228,0.281112,0.001493,0.435897
7,benchmark_agnostic_stress,232,-0.001731,0.375404,-0.000940,0.461207


## Reusable Inference, Checkpoint, And Artifact Reload Test

In [14]:
def make_inference_sequences(feature_frame: pd.DataFrame, X: np.ndarray, seq_len: int) -> tuple[np.ndarray, pd.DataFrame]:
    Xs: list[np.ndarray] = []
    meta: list[dict[str, object]] = []
    working = feature_frame.sort_values(["ticker", "date"]).reset_index(drop=True)
    for ticker, group in working.groupby("ticker", sort=False):
        positions = group.index.to_numpy(dtype=int)
        if len(positions) < seq_len:
            continue
        dates = pd.to_datetime(group["date"]).to_numpy()
        for end_offset in range(seq_len - 1, len(positions)):
            window_positions = positions[end_offset - seq_len + 1 : end_offset + 1]
            row = group.iloc[end_offset]
            Xs.append(X[window_positions])
            meta.append(
                {
                    "date": pd.Timestamp(dates[end_offset]),
                    "ticker": ticker,
                    "historical_vol_20d": float(np.clip(row["vol_20d"] * np.sqrt(252.0), 1e-4, None)),
                    "spy_vol_20d_ann": float(row["spy_vol_20d_ann"]),
                    "spy_drawdown_60d": float(row["spy_drawdown_60d"]),
                    "spy_momentum_20d": float(row["spy_momentum_20d"]),
                    "beta_60d_spy": float(row["beta_60d_spy"]),
                    "benchmark_agnostic_stress": float(row["benchmark_agnostic_stress"]),
                    "avg_pairwise_corr_20d": float(row["avg_pairwise_corr_20d"]),
                }
            )
    if not Xs:
        return np.empty((0, seq_len, len(ALL_FEATURE_NAMES)), dtype=np.float32), pd.DataFrame(meta)
    return np.stack(Xs).astype(np.float32), pd.DataFrame(meta)


def _series_from_mapping(value) -> pd.Series:
    return value if isinstance(value, pd.Series) else pd.Series(value, dtype=float)


def predict_from_prices(model_bundle: dict, prices: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    model_obj = model_bundle["model"]
    model_device = model_bundle.get("device", device)
    feature_names = list(model_bundle["feature_names"])
    seq_len = int(model_bundle["seq_len"])
    horizon = int(model_bundle["horizon"])
    train_mean = _series_from_mapping(model_bundle["train_means"])
    train_std = _series_from_mapping(model_bundle["train_stds"])
    t_means = dict(model_bundle["target_means"])
    t_stds = dict(model_bundle["target_stds"])

    features = build_model_features(prices).replace([np.inf, -np.inf], np.nan)
    if tickers is not None:
        tickers_normalized = [str(ticker).upper() for ticker in tickers]
        features = features.loc[features["ticker"].isin(tickers_normalized)].copy()
    features = features.dropna(subset=feature_names).sort_values(["ticker", "date"]).reset_index(drop=True)
    if features.empty:
        return pd.DataFrame(columns=["date", "ticker", "horizon", "expected_return"])

    X = ((features[feature_names] - train_mean.loc[feature_names]) / train_std.loc[feature_names]).to_numpy(dtype=np.float32)
    X_infer, meta = make_inference_sequences(features, X, seq_len)
    if len(meta) == 0:
        return pd.DataFrame(columns=["date", "ticker", "horizon", "expected_return"])

    model_obj.eval()
    returns, alphas, vols, tails, downside_probs, regime_probs, latent_norms = [], [], [], [], [], [], []
    with torch.no_grad():
        for start in range(0, len(X_infer), 4096):
            xb = torch.as_tensor(X_infer[start : start + 4096], dtype=torch.float32, device=model_device)
            output = model_obj(xb)
            returns.append(output["expected_return"].detach().cpu().numpy())
            alphas.append(output["expected_alpha"].detach().cpu().numpy())
            vols.append(output["expected_vol_log"].detach().cpu().numpy())
            tails.append(output["expected_tail_log"].detach().cpu().numpy())
            downside_probs.append(torch.sigmoid(output["downside_logit"]).detach().cpu().numpy())
            regime_probs.append(torch.softmax(output["regime_logits"], dim=1).detach().cpu().numpy())
            latent_norms.append(torch.linalg.norm(output["latent"], dim=1).detach().cpu().numpy())

    scored = meta.copy()
    scored["expected_return"] = np.concatenate(returns) * t_stds["return"] + t_means["return"]
    scored["expected_alpha"] = np.concatenate(alphas) * t_stds["alpha"] + t_means["alpha"]
    scored["expected_forward_volatility"] = np.expm1(np.concatenate(vols) * t_stds["vol_log"] + t_means["vol_log"]).clip(min=1e-4)
    scored["expected_tail_loss"] = np.expm1(np.concatenate(tails) * t_stds["tail_log"] + t_means["tail_log"]).clip(min=0.0)
    scored["expected_downside_risk"] = np.concatenate(downside_probs)
    scored["expected_volatility"] = np.clip(scored["historical_vol_20d"].to_numpy(dtype=float), 1e-4, None)
    scored["uncertainty"] = np.abs(scored["expected_return"] - scored["expected_alpha"])
    regime_array = np.concatenate(regime_probs)
    for idx, name in enumerate(model_bundle["regime_classes"]):
        scored[f"regime_prob_{name}"] = regime_array[:, idx]
    scored["regime_confidence"] = scored["regime_prob_calm_risk_on"] + 0.50 * scored["regime_prob_rebound"] - 0.75 * scored["regime_prob_high_vol"] - scored["regime_prob_drawdown"]
    scored = add_scores(scored)
    scored["horizon"] = horizon

    if dates is not None:
        requested_dates = pd.to_datetime(pd.Series(dates), utc=True).dt.tz_localize(None)
        scored = scored.loc[scored["date"].isin(set(requested_dates))].copy()

    keep_cols = [
        "date", "ticker", "horizon", "expected_return", "expected_alpha", "expected_volatility", "expected_forward_volatility",
        "expected_tail_loss", "expected_downside_risk", "uncertainty", "regime_confidence",
        "regime_prob_calm_risk_on", "regime_prob_high_vol", "regime_prob_drawdown", "regime_prob_rebound",
        "spy_vol_20d_ann", "spy_drawdown_60d", "spy_momentum_20d", "beta_60d_spy", "benchmark_agnostic_stress",
        "alpha_hist_vol_score", "alpha_pred_vol_score", "vol_blended_score", "strong_downside_score",
        "alpha_hist_vol_score_rank", "alpha_pred_vol_score_rank", "vol_blended_score_rank", "strong_downside_score_rank",
    ]
    return scored.loc[:, keep_cols].reset_index(drop=True)


model_config = {
    "architecture": "VolatilityFocusedLSTMAutoencoder",
    "input_dim": len(ALL_FEATURE_NAMES),
    "hidden_size": HIDDEN_SIZE,
    "latent_dim": LATENT_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "regime_classes": len(REGIME_CLASSES),
    "seq_len": SEQ_LEN,
}
checkpoint = {
    "state_dict": model.state_dict(),
    "model_config": model_config,
    "feature_names": ALL_FEATURE_NAMES,
    "base_feature_names": BASE_FEATURE_NAMES,
    "custom_feature_names": CUSTOM_FEATURE_NAMES,
    "train_means": train_means.to_dict(),
    "train_stds": train_stds.to_dict(),
    "target_means": target_means,
    "target_stds": target_stds,
    "regime_classes": REGIME_CLASSES,
    "regime_thresholds": regime_thresholds,
    "reconstruction_feature_weights": reconstruction_feature_weights.to_dict(),
    "research_end": str(RESEARCH_END.date()),
    "selected_candidate": selected_candidate_name,
    "scoring_config": {
        "primary_score": "strong_downside_score",
        "score_variants": score_columns,
        "predicted_forward_vol_is_denominator": True,
        "downside_score_penalty": DOWNSIDE_SCORE_PENALTY,
        "tail_score_penalty": TAIL_SCORE_PENALTY,
        "uncertainty_score_penalty": UNCERTAINTY_SCORE_PENALTY,
    },
    "portfolio_config": {
        "portfolio_builder": "weights_from_volatility_controlled_scores",
        "selected_candidate": selected_candidate_name,
        "normal_max_weight": NORMAL_MAX_WEIGHT,
        "high_vol_max_weight_options": HIGH_VOL_MAX_WEIGHT_OPTIONS,
        "high_risk_min_active_names": HIGH_RISK_MIN_ACTIVE_NAMES,
        "normal_min_active_names": NORMAL_MIN_ACTIVE_NAMES,
        "turnover_blend": TURNOVER_BLEND,
        "beta_target": BETA_TARGET,
        "minvar_anchor": True,
        "cov_lookback_days": COV_LOOKBACK_DAYS,
    },
    "horizon": HORIZON,
    "rebalance_frequency": REBALANCE_FREQUENCY,
    "random_seed": RANDOM_SEED,
}
model_artifact_path = output_dir / "supervised_lstm_autoencoder_volatility.pt"
torch.save(checkpoint, model_artifact_path)
print("Saved model checkpoint:", model_artifact_path)

model_bundle = {
    "model": model,
    "device": device,
    "feature_names": ALL_FEATURE_NAMES,
    "train_means": train_means,
    "train_stds": train_stds,
    "target_means": target_means,
    "target_stds": target_stds,
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
    "regime_classes": REGIME_CLASSES,
}

weekly_dates = pd.DatetimeIndex(validated_weights.index)
submission_style_predictions = predict_from_prices(model_bundle, prices, dates=weekly_dates, tickers=validated_weights.columns)
submission_style_predictions = validate_prediction_frame(submission_style_predictions, dataset_name=research_spec, horizon=HORIZON, repo_root=repo_root)


def load_model_bundle_from_checkpoint(checkpoint_path: Path, map_location="cpu") -> dict[str, object]:
    payload = torch.load(checkpoint_path, map_location=map_location)
    cfg = dict(payload["model_config"])
    model_obj = VolatilityFocusedLSTMAutoencoder(
        input_dim=cfg["input_dim"],
        hidden_size=cfg["hidden_size"],
        latent_dim=cfg["latent_dim"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        regime_classes=cfg["regime_classes"],
    )
    model_obj.load_state_dict(payload["state_dict"])
    model_obj.eval()
    return {
        "model": model_obj,
        "device": torch.device(map_location),
        "feature_names": payload["feature_names"],
        "train_means": pd.Series(payload["train_means"], dtype=float),
        "train_stds": pd.Series(payload["train_stds"], dtype=float),
        "target_means": payload["target_means"],
        "target_stds": payload["target_stds"],
        "seq_len": payload["model_config"]["seq_len"],
        "horizon": payload["horizon"],
        "regime_classes": payload["regime_classes"],
    }


reloaded_bundle = load_model_bundle_from_checkpoint(model_artifact_path, map_location="cpu")
reload_predictions = predict_from_prices(reloaded_bundle, prices, dates=weekly_dates[: min(3, len(weekly_dates))], tickers=validated_weights.columns)
reload_predictions = validate_prediction_frame(reload_predictions, dataset_name=research_spec, horizon=HORIZON, repo_root=repo_root)
print("Reusable inference predictions:", submission_style_predictions.shape)
print("Artifact reload predictions:", reload_predictions.shape)
display(submission_style_predictions.head())

Saved model checkpoint: /content/Portfolio-Optimization-Lib/runs/supervised_lstm_autoencoder_volatility/supervised_lstm_autoencoder_volatility.pt
Reusable inference predictions: (1275, 28)
Artifact reload predictions: (75, 28)


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,expected_forward_volatility,expected_tail_loss,expected_downside_risk,uncertainty,...,beta_60d_spy,benchmark_agnostic_stress,alpha_hist_vol_score,alpha_pred_vol_score,vol_blended_score,strong_downside_score,alpha_hist_vol_score_rank,alpha_pred_vol_score_rank,vol_blended_score_rank,strong_downside_score_rank
0,2022-01-07,AAPL,5,0.005275,0.004988,0.291345,0.219727,0.020037,0.152104,0.000288,...,1.164415,0.765577,0.017119,0.022699,-0.188023,-0.604908,-0.44,-0.28,0.60,0.36
1,2022-01-07,ADBE,5,0.010866,0.009812,0.519157,0.305341,0.024806,0.258852,0.001054,...,2.093541,0.765577,0.018901,0.032136,-0.264365,-0.702027,-0.20,0.20,-0.68,-0.84
2,2022-01-07,ADI,5,0.010983,0.010342,0.242832,0.245846,0.017838,0.175376,0.000641,...,1.297955,0.765577,0.042590,0.042068,-0.178156,-0.590746,0.68,0.44,0.76,0.60
3,2022-01-07,AMAT,5,0.011979,0.011773,0.408458,0.378215,0.030387,0.185531,0.000207,...,1.853227,0.765577,0.028822,0.031127,-0.217259,-0.660968,0.28,0.12,-0.20,-0.28
4,2022-01-07,AMD,5,0.012489,0.016148,0.588302,0.435677,0.034362,0.241010,0.003659,...,2.407290,0.765577,0.027448,0.037064,-0.256848,-0.693786,0.20,0.36,-0.60,-0.60


## MLflow Logging

In [15]:
def mlflow_tracking_preflight(tracking_uri: str, timeout: float = 2.0) -> tuple[bool, str]:
    from urllib.error import HTTPError, URLError
    from urllib.parse import urlparse
    from urllib.request import Request, urlopen
    import socket

    parsed = urlparse(tracking_uri)
    if parsed.scheme not in {"http", "https"}:
        return True, "local tracking URI"
    if not parsed.hostname:
        return False, f"remote tracking URI has no hostname: {tracking_uri}"
    proxy_env = ["HTTPS_PROXY", "HTTP_PROXY", "ALL_PROXY", "https_proxy", "http_proxy", "all_proxy"]
    configured_proxy = next((name for name in proxy_env if os.environ.get(name)), None)
    if configured_proxy:
        try:
            request = Request(tracking_uri.rstrip("/") + "/", method="GET")
            with urlopen(request, timeout=timeout):
                return True, f"{parsed.hostname} reachable through {configured_proxy}"
        except HTTPError as exc:
            return True, f"{parsed.hostname} reachable through {configured_proxy}; HTTP {exc.code}"
        except URLError as exc:
            return False, f"{parsed.hostname} not reachable through {configured_proxy} ({exc.reason})"
        except OSError as exc:
            return False, f"{parsed.hostname} not reachable through {configured_proxy} ({exc})"
    port = parsed.port or (443 if parsed.scheme == "https" else 80)
    try:
        with socket.create_connection((parsed.hostname, port), timeout=timeout):
            return True, f"{parsed.hostname}:{port} reachable"
    except OSError as exc:
        return False, f"{parsed.hostname}:{port} not reachable ({exc})"


if LOG_TO_MLFLOW:
    mlflow_layout = init_mlflow(repo_root)
    tracking_uri = mlflow_layout["tracking_uri"]
    print("MLflow tracking URI:", tracking_uri)
    tracking_ok, tracking_reason = mlflow_tracking_preflight(tracking_uri)
    if not tracking_ok:
        print("MLflow logging skipped:", tracking_reason)
    else:
        try:
            with start_run(
                run_name=RUN_NAME,
                dataset_name=DATASET_NAME,
                tags={
                    "workflow": "volatility_focused_supervised_lstm_autoencoder",
                    "model_family": "torch_lstm_autoencoder",
                    "prediction_horizon": str(HORIZON),
                    "rebalance_frequency": REBALANCE_FREQUENCY,
                    "research_end": str(RESEARCH_END.date()),
                },
                repo_root=repo_root,
            ):
                import mlflow
                mlflow.log_params(
                    {
                        "model_name": MODEL_NAME,
                        "dataset_name": DATASET_NAME,
                        "research_end": str(RESEARCH_END.date()),
                        "horizon": HORIZON,
                        "seq_len": SEQ_LEN,
                        "feature_count": len(ALL_FEATURE_NAMES),
                        "latent_dim": LATENT_DIM,
                        "hidden_size": HIDDEN_SIZE,
                        "num_layers": NUM_LAYERS,
                        "dropout": DROPOUT,
                        "epochs_requested": EPOCHS,
                        "epochs_run": len(history),
                        "selected_candidate": selected_candidate_name,
                        "portfolio_score": portfolio.metadata["score_col"],
                        "portfolio_high_vol_max_weight": portfolio.metadata["high_vol_max_weight"],
                        "portfolio_beta_target": BETA_TARGET,
                        "avg_active_names": float((validated_weights > 1e-8).sum(axis=1).mean()),
                        "avg_effective_names": float(risk_log["effective_names"].mean()),
                        "avg_realized_max_weight": float(risk_log["realized_max_weight"].mean()),
                        "avg_beta_estimate": float(risk_log["portfolio_beta_estimate"].mean()),
                        "val_return_mse": val_return_mse,
                        "val_alpha_mse": val_alpha_mse,
                        "val_vol_mse": val_vol_mse,
                        "val_tail_mse": val_tail_mse,
                        "val_downside_brier": val_downside_brier,
                        "val_regime_accuracy": val_regime_accuracy,
                    }
                )
                for _, row in final_stress_table.iterrows():
                    mlflow.log_metric(f"stress_{row['stress_metric']}", float(row["value"]))
                log_predictions(predictions)
                log_portfolio(portfolio)
                log_backtest(result)
                manifest = log_model_submission(
                    {"model_checkpoint": model_artifact_path},
                    model_name=MODEL_NAME,
                    model_family="torch",
                    feature_names=ALL_FEATURE_NAMES,
                    target=alpha_target_col,
                    horizon=HORIZON,
                    rebalance_frequency=REBALANCE_FREQUENCY,
                    preprocessing={
                        "scaler": "train_mean_std_non_benchmark_rows_through_2022",
                        "target_scaler": "return_alpha_forward_vol_tail_mean_std_non_benchmark_rows",
                        "train_means": train_means.to_dict(),
                        "train_stds": train_stds.to_dict(),
                        "target_means": target_means,
                        "target_stds": target_stds,
                        "regime_thresholds": regime_thresholds,
                        "regime_classes": REGIME_CLASSES,
                    },
                    model_config={
                        **model_config,
                        "portfolio_builder": "weights_from_volatility_controlled_scores",
                        "required_functions": ["build_model_features", "predict_from_prices"],
                        "portfolio": portfolio.metadata,
                        "scoring_config": checkpoint["scoring_config"],
                        "artifact_reload_test": "passed",
                    },
                    source_files=[NOTEBOOK_PATH] if NOTEBOOK_PATH.exists() else None,
                    notes="Volatility-focused LSTM-AE using predicted forward volatility as a denominator, downside/tail risk heads, min-variance anchoring, high-vol max-weight caps, beta targeting, and pre-2023 volatile-regime tests.",
                )
                print("MLflow logging complete. Submission manifest keys:", sorted(manifest.keys()))
        except Exception as exc:
            print("MLflow logging skipped after error:", repr(exc))
else:
    print("MLflow logging skipped because SKIP_MLFLOW=1.")

MLflow tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
MLflow logging skipped: adams-macbook-pro.tail5ddc35.ts.net:443 not reachable ([Errno -2] Name or service not known)


## Final Checks

In [16]:
assert {"total_return", "annual_return", "annual_volatility", "sharpe", "max_drawdown"}.issubset(result.metrics)
assert pd.Timestamp(prices["date"].max()) <= RESEARCH_END, "Local research data should not include dates after 2022."
assert set(validated_weights.columns).isdisjoint({research_spec.benchmark_ticker}), "Benchmark ticker should not be traded."
assert validated_weights.index.max() <= RESEARCH_END, "Local backtest weights should not extend beyond the research cap."
assert validated_weights.index.is_monotonic_increasing
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert predictions["expected_forward_volatility"].notna().all()
assert predictions["expected_tail_loss"].notna().all()
assert predictions["expected_downside_risk"].between(0.0, 1.0).all()
assert predictions["ticker"].ne(research_spec.benchmark_ticker).all()
assert Path(artifact_paths["quantstats_report"]).exists()
assert model_artifact_path.exists()
loaded_payload = torch.load(model_artifact_path, map_location="cpu")
assert "state_dict" in loaded_payload
assert loaded_payload["feature_names"] == ALL_FEATURE_NAMES
assert len(submission_style_predictions) == len(predictions)
assert len(reload_predictions) > 0

print("End-to-end volatility-focused supervised LSTM autoencoder workflow validated successfully.")
print("Selected candidate:", selected_candidate_name)
print("Key risk metrics:")
for key in ["annual_return", "annual_volatility", "sharpe", "sortino", "max_drawdown", "average_turnover"]:
    print(f"  {key:<20s}: {result.metrics[key]:.6f}")
print("Average active names:", float((validated_weights > 1e-8).sum(axis=1).mean()))
print("Average effective names:", float(risk_log["effective_names"].mean()))
print("Average max weight:", float(risk_log["realized_max_weight"].mean()))
print("Average beta estimate:", float(risk_log["portfolio_beta_estimate"].mean()))

display(result.nav.tail().to_frame("nav"))
display(risk_log.tail())

End-to-end volatility-focused supervised LSTM autoencoder workflow validated successfully.
Selected candidate: weekly_vol_blended_score_hvw01
Key risk metrics:
  annual_return       : -0.303695
  annual_volatility   : 0.371610
  sharpe              : -0.817240
  sortino             : -1.420967
  max_drawdown        : -0.366763
  average_turnover    : 0.049686
Average active names: 24.980392156862745
Average effective names: 22.87151872763332
Average max weight: 0.05045162504604634
Average beta estimate: 1.467084487968458


,nav
2022-12-23,70.018384
2022-12-27,68.838871
2022-12-28,67.953701
2022-12-29,70.119731
2022-12-30,70.132038


,date,score_col,frequency,unfavorable_regime,effective_max_weight,selected_names,active_names,effective_names,realized_max_weight,portfolio_beta_estimate,beta_blend,tilt_strength,turnover_estimate,proxy_drawdown
46,2022-11-25,vol_blended_score_rank,weekly,True,0.03,25,25,25.0,0.040001,1.392392,0.00,0.025,7.642986e-06,-0.245229
47,2022-12-02,vol_blended_score_rank,weekly,True,0.03,25,25,25.0,0.040000,1.418380,0.85,0.025,4.203642e-06,-0.230327
48,2022-12-09,vol_blended_score_rank,weekly,True,0.03,25,25,25.0,0.040000,1.412355,0.85,0.025,2.312003e-06,-0.256989
49,2022-12-16,vol_blended_score_rank,weekly,True,0.03,25,25,25.0,0.040000,1.425322,0.00,0.025,1.271602e-06,-0.277775
50,2022-12-22,vol_blended_score_rank,weekly,True,0.03,25,25,25.0,0.040000,1.455368,0.00,0.025,6.993810e-07,-0.301042
